Dans cette étape, on distinguera :
{Profil + T0 == recommandation de parcours DA/BI/DS/IA}

puis, après T1 :{Profil + T0 + T1}== {niveau + lacunes + prérequis}=={groupes pédagogiques}

Phase A — Construire les populations de recommandation
Phase B — Recommandation de parcours DA / BI / DS / IA
Phase C — Détermination du niveau Débutant / Intermédiaire / Avancé
Phase D — Lacunes et prérequis
Phase E — Construction des groupes pédagogiques
Phase F — Validation des recommandations + HITL
Phase G — Exports vers 06_RAG_LLM_HITL.ipynb

Cellule Markdown — Titre
# 05 — Groupes et recommandation pédagogique

## Objectifs

Cette étape construit les mécanismes de recommandation pédagogique à
partir des résultats des étapes précédentes.

Deux moments sont distingués :

\[
R_{T0}: Profil + T0 \rightarrow Parcours
\]

pour recommander un parcours parmi DA, BI, DS et IA ;

et

\[
R_{T1}: Profil + T0 + T1 + \widehat{T2}
\rightarrow Niveau + Lacunes + Prérequis + Groupe
\]

pour constituer des groupes pédagogiques homogènes.

Les recommandations sont produites pour des groupes d'apprenants
présentant des besoins compatibles. Elles ne constituent pas une
décision pédagogique individuelle automatique.

La décision finale reste soumise à validation humaine.

In [1]:
# ============================================================
# CELLULE 1 — Imports
# ============================================================

from pathlib import Path

import json
import numpy as np
import pandas as pd

from IPython.display import display

import joblib


RANDOM_STATE = 42

print("✅ Imports chargés")

✅ Imports chargés


# Cellule 2 — Répertoires

In [2]:
# ============================================================
# CELLULE 2 — Répertoires
# ============================================================

def trouver_racine_projet():

    candidats = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]

    for racine in candidats:

        if (
            racine
            / "outputs"
            / "01_nettoyage"
        ).exists():

            return racine.resolve()

    raise FileNotFoundError(
        "Répertoire outputs/01_nettoyage introuvable."
    )


PROJECT_ROOT = trouver_racine_projet()


INPUT_01 = (
    PROJECT_ROOT
    / "outputs"
    / "01_nettoyage"
)

INPUT_02 = (
    PROJECT_ROOT
    / "outputs"
    / "02_diagnostic_T0"
)

INPUT_03 = (
    PROJECT_ROOT
    / "outputs"
    / "03_modelisation_MT0_MT1"
)

INPUT_04 = (
    PROJECT_ROOT
    / "outputs"
    / "04_validation_XAI"
)


OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "05_groupes_recommandation"
)

TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"


for dossier in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIGURE_DIR
]:

    dossier.mkdir(
        parents=True,
        exist_ok=True
    )


print("PROJECT_ROOT :", PROJECT_ROOT)
print("OUTPUT 05    :", OUTPUT_DIR)

PROJECT_ROOT : /home/andrycomputer/Documents/Model-eval-pers-elearning
OUTPUT 05    : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/05_groupes_recommandation


# Cellule 3 — Charger les sources utiles

In [3]:
# ============================================================
# CELLULE 3 — Chargement
# ============================================================

D1 = pd.read_csv(
    INPUT_01
    / "D1_profils_nettoyes.csv"
)


T0_SCORES = pd.read_csv(
    INPUT_02
    / "T0_scores_domaines.csv"
)


DIAG_T0 = pd.read_csv(
    INPUT_02
    / "diagnostic_initial_T0.csv"
)


BASE_MT1_125 = pd.read_csv(
    INPUT_03
    / "base_modelisation_MT1_N125.csv"
)


OOF_MT1_125 = pd.read_csv(
    INPUT_03
    / "oof_MT1_N125_agrege.csv"
)


T1_SCORING_LONG = pd.read_csv(
    INPUT_03
    / "T1_scoring_long.csv"
)


print("D1          :", D1.shape)
print("T0 scores   :", T0_SCORES.shape)
print("Diagnostic  :", DIAG_T0.shape)
print("Base MT1    :", BASE_MT1_125.shape)

print(
    "OOF M_T1 N125 :",
    OOF_MT1_125.shape
)

print(
    "T1 scoring long :",
    T1_SCORING_LONG.shape
)

print(
    "\n✅ Données chargées"
)

D1          : (472, 22)
T0 scores   : (305, 5)
Diagnostic  : (305, 24)
Base MT1    : (125, 52)
OOF M_T1 N125 : (875, 10)
T1 scoring long : (2940, 17)

✅ Données chargées


# Cellule 4 — Population disponible pour la recommandation de parcours
Ici on utilise uniquement Profil + T0.

In [4]:
# ============================================================
# CELLULE 4 — Cohorte R_T0 : Profil + T0
# ============================================================

ID = "_ID_NORM"


assert D1[ID].is_unique
assert T0_SCORES[ID].is_unique


BASE_RECO_T0 = (
    D1
    .merge(
        T0_SCORES,
        on=ID,
        how="inner",
        validate="one_to_one"
    )
)


print(
    "Apprenants avec Profil + T0 :",
    len(BASE_RECO_T0)
)

print(
    "IDs uniques :",
    BASE_RECO_T0[ID].nunique()
)


assert (
    len(BASE_RECO_T0)
    ==
    BASE_RECO_T0[ID].nunique()
)


print(
    "\n✅ Cohorte initiale de recommandation créée"
)

Apprenants avec Profil + T0 : 305
IDs uniques : 305

✅ Cohorte initiale de recommandation créée


# cellule 5 — Vérifier les quatre diagnostics T0

In [5]:
# ============================================================
# CELLULE 5 — Vérification des scores par domaine
# ============================================================

COLONNES_T0 = [
    "S_DA",
    "S_BI",
    "S_DS",
    "S_IA"
]


print(
    BASE_RECO_T0[
        COLONNES_T0
    ]
    .describe()
    .T
)


print(
    "\nValeurs manquantes :"
)

display(
    BASE_RECO_T0[
        COLONNES_T0
    ]
    .isna()
    .sum()
    .rename("N_MANQUANT")
    .to_frame()
)

      count       mean        std  min  25%   50%   75%    max
S_DA  305.0  27.606557  27.953956  0.0  0.0  20.0  40.0  100.0
S_BI  305.0  21.573770  27.244301  0.0  0.0  20.0  40.0  100.0
S_DS  305.0  21.508197  27.416514  0.0  0.0  20.0  40.0  100.0
S_IA  305.0  20.852459  27.967225  0.0  0.0   0.0  40.0  100.0

Valeurs manquantes :


,N_MANQUANT
S_DA,0
S_BI,0
S_DS,0
S_IA,0


# Cellule 6 — Inventaire des informations pédagogiques

Avant de créer une règle de recommandation, nous devons regarder exactement les modalités disponibles.

In [6]:
# ============================================================
# CELLULE 6 — Inventaire profil / objectifs / prérequis
# ============================================================

colonnes_auditer = [

    "Objectif pro",

    "Niveau informatique",

    "Niveau Excel",

    "Niveau Power BI",

    "Niveau Python",

    "Niveau IA",

    "Filière",

    "Niveau d'étude",

    "Statut actuel",

    "Motivation",

    "Disponibilité"
]


for col in colonnes_auditer:

    if col not in BASE_RECO_T0.columns:
        continue

    print(
        "\n",
        "=" * 70
    )

    print(col)

    print("=" * 70)

    display(
        BASE_RECO_T0[
            col
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(col)
        .reset_index(
            name="N"
        )
    )


Objectif pro


,Objectif pro,N
0,Entrepreneur,97
1,Developpeur Python / IA,58
2,Data Analyst,57
3,Business Analyst,38
4,Non defini,29
5,Consultant,14
6,Data Scientist,11
7,NaN,1



Niveau informatique


,Niveau informatique,N
0,Intermediaire,191
1,Debutant,92
2,Aucun,12
3,Avance,10



Niveau Excel


...

,Statut actuel,N
0,Etudiant(e) a plein temps,212
1,Etudiant(e) en fin d etudes,34
2,Diplome(e) en recherche d emploi,20
3,Etudiant(e) en alternance,19
4,Etudiant(e) en stage,15
5,Professionnel(le) en activite,5



Motivation


,Motivation,N
0,Acquerir des competences techniques,63
1,Curiosite,24
2,Ameliorer mon employabilite,17
3,Certification,9
4,Acquerir des competences techniques | Ameliore...,8
...,...,...
115,Acquerir des competences techniques | Ameliore...,1
116,Realiser un projet perso | Reconversion,1
117,Acquerir des competences techniques | Curiosit...,1
118,Ameliorer mon employabilite | Acquerir des com...,1



Disponibilité


,Disponibilité,N
0,En semaine - Soir,69
1,Week-end,40
2,En semaine - Apres-midi,33
3,Flexible,23
4,En semaine - Soir | Week-end,21
5,En semaine - Matin,15
6,En semaine - Soir | Flexible,15
7,NaN,12
8,Samedi,11
9,En semaine - Soir | Samedi,6


# Cellule 7 — Vérifier la base M_T1 pour les groupes après T1

In [7]:
# ============================================================
# CELLULE 7 — Population disponible après T1
# ============================================================

colonnes_mt1_importantes = [

    "IDENTIFICATION",

    "T0__S_DA",
    "T0__S_BI",
    "T0__S_DS",
    "T0__S_IA",

    "T1__SCORE",
    "T1__PARCOURS",

    "TARGET_T2"
]


colonnes_mt1_presentes = [
    c
    for c in colonnes_mt1_importantes
    if c in BASE_MT1_125.columns
]


display(
    BASE_MT1_125[
        colonnes_mt1_presentes
    ].head(10)
)


print(
    "\nPopulation expérimentale après T1 :",
    len(BASE_MT1_125)
)

,T0__S_DA,T0__S_BI,T0__S_DS,T0__S_IA,T1__SCORE,T1__PARCOURS,TARGET_T2
0,0.0,0.0,0.0,0.0,93.333333,IA,93.333333
1,40.0,20.0,80.0,20.0,86.666667,IA,80.000000
2,20.0,0.0,20.0,40.0,33.333333,BI,100.000000
3,60.0,40.0,20.0,20.0,73.333333,IA,93.333333
4,0.0,0.0,20.0,20.0,80.000000,IA,53.333333
5,40.0,40.0,0.0,0.0,46.666667,DS,86.666667
6,20.0,0.0,0.0,0.0,53.333333,BI,60.000000
7,20.0,0.0,0.0,0.0,86.666667,BI,60.000000
8,20.0,0.0,20.0,0.0,60.000000,BI,80.000000
9,20.0,0.0,0.0,0.0,80.000000,IA,73.333333



Population expérimentale après T1 : 125


# Cellule 8 — Résumé des deux populations

In [8]:
# ============================================================
# CELLULE 8 — Résumé Phase A
# ============================================================

resume_populations = pd.DataFrame({

    "POPULATION": [
        "R_T0",
        "R_T1 expérimental"
    ],

    "DONNEES_DISPONIBLES": [
        "Profil + T0",
        "Profil + T0 + T1"
    ],

    "N": [
        len(BASE_RECO_T0),
        len(BASE_MT1_125)
    ],

    "UTILISATION": [
        "Recommandation initiale de parcours",
        "Niveau, lacunes, prérequis et groupes"
    ]

})


display(
    resume_populations
)

,POPULATION,DONNEES_DISPONIBLES,N,UTILISATION
0,R_T0,Profil + T0,305,Recommandation initiale de parcours
1,R_T1 expérimental,Profil + T0 + T1,125,"Niveau, lacunes, prérequis et groupes"


la Phase B — recommandation de parcours.
Point important : nous avons bien 305 apprenants avec Profil + T0. C’est donc cette population qui sert à la recommandation initiale de parcours.
Je vous propose une règle sans poids arbitraires :
1. priorité au meilleur score T0 parmi DA/BI/DS/IA ;
2. en cas d’égalité, utiliser l’objectif professionnel ;
3. s’il reste une égalité, utiliser les prérequis déclarés ;
4. si l’égalité persiste, ne pas forcer : A_VALIDER.

# Cellule 9 — Encoder les niveaux déclarés

In [9]:
# ============================================================
# CELLULE 9 — Encodage des niveaux déclarés
# ============================================================

NIVEAU_ORDINAL = {
    "Aucun": 0,
    "Debutant": 1,
    "Intermediaire": 2,
    "Avance": 3
}


BASE_RECO_T0["LVL_INFORMATIQUE"] = (
    BASE_RECO_T0["Niveau informatique"]
    .map(NIVEAU_ORDINAL)
)

BASE_RECO_T0["LVL_EXCEL"] = (
    BASE_RECO_T0["Niveau Excel"]
    .map(NIVEAU_ORDINAL)
)

BASE_RECO_T0["LVL_POWER_BI"] = (
    BASE_RECO_T0["Niveau Power BI"]
    .map(NIVEAU_ORDINAL)
)

BASE_RECO_T0["LVL_PYTHON"] = (
    BASE_RECO_T0["Niveau Python"]
    .map(NIVEAU_ORDINAL)
)

BASE_RECO_T0["LVL_IA"] = (
    BASE_RECO_T0["Niveau IA"]
    .map(NIVEAU_ORDINAL)
)


print("✅ Niveaux déclarés encodés")

✅ Niveaux déclarés encodés


# Cellule 10 — Alignement objectif professionnel → parcours
On ne mappe que les objectifs explicitement liés à un parcours.

In [10]:
# ============================================================
# CELLULE 10 — Mapping objectif professionnel
# ============================================================

OBJECTIF_PARCOURS = {

    "Data Analyst":
        "DA",

    "Business Analyst":
        "BI",

    "Data Scientist":
        "DS",

    "Developpeur Python / IA":
        "IA"
}


BASE_RECO_T0["PARCOURS_OBJECTIF"] = (
    BASE_RECO_T0["Objectif pro"]
    .map(OBJECTIF_PARCOURS)
)


display(
    BASE_RECO_T0[
        [
            "Objectif pro",
            "PARCOURS_OBJECTIF"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Objectif pro",
        na_position="last"
    )
)

,Objectif pro,PARCOURS_OBJECTIF
4,Business Analyst,BI
12,Consultant,NaN
7,Data Analyst,DA
0,Data Scientist,DS
1,Developpeur Python / IA,IA
2,Entrepreneur,NaN
41,Non defini,NaN
19,NaN,NaN


# Cellule 11 — Préparation des prérequis déclarés
Ce mapping est utilisé uniquement comme départage secondaire, pas comme vérité absolue.

In [11]:
# ============================================================
# CELLULE 11 — Readiness / prérequis déclarés
# ============================================================

BASE_RECO_T0["READY_DA"] = (
    BASE_RECO_T0[
        [
            "LVL_INFORMATIQUE",
            "LVL_EXCEL"
        ]
    ]
    .mean(axis=1)
)


BASE_RECO_T0["READY_BI"] = (
    BASE_RECO_T0[
        [
            "LVL_INFORMATIQUE",
            "LVL_EXCEL",
            "LVL_POWER_BI"
        ]
    ]
    .mean(axis=1)
)


BASE_RECO_T0["READY_DS"] = (
    BASE_RECO_T0[
        [
            "LVL_INFORMATIQUE",
            "LVL_PYTHON"
        ]
    ]
    .mean(axis=1)
)


BASE_RECO_T0["READY_IA"] = (
    BASE_RECO_T0[
        [
            "LVL_INFORMATIQUE",
            "LVL_PYTHON",
            "LVL_IA"
        ]
    ]
    .mean(axis=1)
)


print("✅ Indicateurs de prérequis calculés")

✅ Indicateurs de prérequis calculés


Cellule 12 — Fonction de recommandation

In [12]:
# ============================================================
# CELLULE 12 — Recommandation de parcours
# ============================================================

PARCOURS = [
    "DA",
    "BI",
    "DS",
    "IA"
]


SCORE_COL = {
    "DA": "S_DA",
    "BI": "S_BI",
    "DS": "S_DS",
    "IA": "S_IA"
}


READY_COL = {
    "DA": "READY_DA",
    "BI": "READY_BI",
    "DS": "READY_DS",
    "IA": "READY_IA"
}


def recommander_parcours(row):

    # --------------------------------------------------------
    # 1. Scores T0
    # --------------------------------------------------------

    scores = {
        p: float(row[SCORE_COL[p]])
        for p in PARCOURS
    }


    score_max = max(
        scores.values()
    )


    candidats = [
        p
        for p in PARCOURS
        if scores[p] == score_max
    ]


    # --------------------------------------------------------
    # Cas 1 : meilleur T0 unique
    # --------------------------------------------------------

    if len(candidats) == 1:

        choisi = candidats[0]

        return pd.Series({
            "PARCOURS_RECOMMANDE":
                choisi,

            "PARCOURS_CANDIDATS":
                choisi,

            "REGLE_DECISION":
                "T0_unique",

            "SCORE_T0_RECOMMANDE":
                score_max,

            "NB_CANDIDATS":
                1
        })


    # --------------------------------------------------------
    # 2. Départage par objectif professionnel
    # --------------------------------------------------------

    parcours_objectif = row[
        "PARCOURS_OBJECTIF"
    ]


    if (
        pd.notna(parcours_objectif)
        and
        parcours_objectif in candidats
    ):

        return pd.Series({
            "PARCOURS_RECOMMANDE":
                parcours_objectif,

            "PARCOURS_CANDIDATS":
                " | ".join(candidats),

            "REGLE_DECISION":
                "T0_egalite_objectif",

            "SCORE_T0_RECOMMANDE":
                score_max,

            "NB_CANDIDATS":
                len(candidats)
        })


    # --------------------------------------------------------
    # 3. Départage par prérequis déclarés
    # --------------------------------------------------------

    readiness = {
        p: row[READY_COL[p]]
        for p in candidats
    }


    readiness_max = max(
        readiness.values()
    )


    candidats_ready = [
        p
        for p in candidats
        if readiness[p] == readiness_max
    ]


    if len(candidats_ready) == 1:

        choisi = candidats_ready[0]

        return pd.Series({
            "PARCOURS_RECOMMANDE":
                choisi,

            "PARCOURS_CANDIDATS":
                " | ".join(candidats),

            "REGLE_DECISION":
                "T0_egalite_prerequis",

            "SCORE_T0_RECOMMANDE":
                score_max,

            "NB_CANDIDATS":
                len(candidats)
        })


    # --------------------------------------------------------
    # 4. Ne pas forcer une décision
    # --------------------------------------------------------

    return pd.Series({
        "PARCOURS_RECOMMANDE":
            "A_VALIDER",

        "PARCOURS_CANDIDATS":
            " | ".join(candidats_ready),

        "REGLE_DECISION":
            "egalite_persistante",

        "SCORE_T0_RECOMMANDE":
            score_max,

        "NB_CANDIDATS":
            len(candidats_ready)
    })

# Cellule 13 — Appliquer la recommandation aux 305 apprenants

In [13]:
# ============================================================
# CELLULE 13 — Application sur N=305
# ============================================================

RESULTATS_RECO = (
    BASE_RECO_T0
    .apply(
        recommander_parcours,
        axis=1
    )
)


BASE_RECO_T0 = pd.concat(
    [
        BASE_RECO_T0,
        RESULTATS_RECO
    ],
    axis=1
)


print(
    "N recommandations :",
    len(BASE_RECO_T0)
)

print(
    "\nDistribution des parcours recommandés :"
)


display(
    BASE_RECO_T0[
        "PARCOURS_RECOMMANDE"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "PARCOURS_RECOMMANDE"
    )
    .reset_index(
        name="N"
    )
)

N recommandations : 305

Distribution des parcours recommandés :


,PARCOURS_RECOMMANDE,N
0,DA,124
1,IA,54
2,BI,48
3,DS,43
4,A_VALIDER,36


# Cellule 14 — Voir comment les décisions ont été prises

In [14]:
# ============================================================
# CELLULE 14 — Règles de décision utilisées
# ============================================================

resume_regles = (
    BASE_RECO_T0[
        "REGLE_DECISION"
    ]
    .value_counts()
    .rename_axis(
        "REGLE_DECISION"
    )
    .reset_index(
        name="N"
    )
)


resume_regles[
    "POURCENTAGE"
] = (
    resume_regles["N"]
    /
    len(BASE_RECO_T0)
    * 100
)


display(
    resume_regles.round(2)
)

,REGLE_DECISION,N,POURCENTAGE
0,T0_unique,170,55.74
1,T0_egalite_prerequis,57,18.69
2,T0_egalite_objectif,42,13.77
3,egalite_persistante,36,11.80


# Cellule 15 — Mesurer l’ambiguïté T0
Comme T0 n’a que 5 questions par domaine, les égalités peuvent être fréquentes.

In [15]:
# ============================================================
# CELLULE 15 — Ambiguïté de la recommandation
# ============================================================

def marge_t0(row):

    scores = sorted(
        [
            row["S_DA"],
            row["S_BI"],
            row["S_DS"],
            row["S_IA"]
        ],
        reverse=True
    )

    return (
        scores[0]
        -
        scores[1]
    )


BASE_RECO_T0[
    "MARGE_T0"
] = (
    BASE_RECO_T0
    .apply(
        marge_t0,
        axis=1
    )
)


resume_marge = (
    BASE_RECO_T0[
        "MARGE_T0"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "MARGE_T0"
    )
    .reset_index(
        name="N"
    )
)


display(
    resume_marge
)

,MARGE_T0,N
0,0.0,135
1,20.0,116
2,40.0,43
3,60.0,8
4,80.0,3


0  → égalité au meilleur score
20 → une question d'écart
40 → deux questions d'écart
...

#  Cellule 16 — Comparaison descriptive avec le parcours observé à T1

Très important : T1__PARCOURS n’est pas considéré comme la vérité absolue. On mesure seulement la concordance avec le parcours réellement observé.

In [16]:
# ============================================================
# CELLULE 16 — Concordance descriptive avec T1
# ============================================================

ID = "_ID_NORM"


validation_reco = (
    BASE_RECO_T0[
        [
            ID,
            "PARCOURS_RECOMMANDE",
            "PARCOURS_CANDIDATS",
            "REGLE_DECISION",
            "MARGE_T0"
        ]
    ]
    .merge(
        BASE_MT1_125[
            [
                ID,
                "T1__PARCOURS"
            ]
        ],
        on=ID,
        how="inner",
        validate="one_to_one"
    )
)


validation_reco[
    "CONCORDANCE_T1"
] = (
    validation_reco[
        "PARCOURS_RECOMMANDE"
    ]
    ==
    validation_reco[
        "T1__PARCOURS"
    ]
)


# Exclure A_VALIDER pour calculer la concordance
validation_decidee = (
    validation_reco[
        validation_reco[
            "PARCOURS_RECOMMANDE"
        ] != "A_VALIDER"
    ]
)


print(
    "N comparaison :",
    len(validation_reco)
)

print(
    "N recommandations décidées :",
    len(validation_decidee)
)

print(
    "Concordance descriptive avec T1 :",
    round(
        validation_decidee[
            "CONCORDANCE_T1"
        ].mean() * 100,
        2
    ),
    "%"
)

N comparaison : 125
N recommandations décidées : 109
Concordance descriptive avec T1 : 21.1 %


# Cellule 17 — Le parcours T1 appartient-il au groupe des meilleurs candidats T0 ?

In [17]:
# ============================================================
# CELLULE 17 — T1 appartient-il aux candidats T0 ?
# ============================================================

def t1_dans_candidats(row):

    candidats = str(
        row["PARCOURS_CANDIDATS"]
    ).split(" | ")

    return (
        row["T1__PARCOURS"]
        in candidats
    )


validation_reco[
    "T1_DANS_CANDIDATS"
] = (
    validation_reco
    .apply(
        t1_dans_candidats,
        axis=1
    )
)


print(
    "T1 présent parmi les candidats T0 :",
    round(
        validation_reco[
            "T1_DANS_CANDIDATS"
        ].mean() * 100,
        2
    ),
    "%"
)

T1 présent parmi les candidats T0 : 44.8 %


# Cellule 18 — Concordance selon la règle utilisée

In [18]:
# ============================================================
# CELLULE 18 — Concordance selon la règle
# ============================================================

concordance_par_regle = (
    validation_reco[
        validation_reco[
            "PARCOURS_RECOMMANDE"
        ] != "A_VALIDER"
    ]
    .groupby(
        "REGLE_DECISION",
        as_index=False
    )
    .agg(
        N=("CONCORDANCE_T1", "size"),
        CONCORDANCE_PCT=(
            "CONCORDANCE_T1",
            lambda x: x.mean() * 100
        )
    )
)


display(
    concordance_par_regle.round(2)
)

,REGLE_DECISION,N,CONCORDANCE_PCT
0,T0_egalite_objectif,20,20.00
1,T0_egalite_prerequis,18,22.22
2,T0_unique,71,21.13


# Cellule 19 — Concordance selon la marge T0

In [19]:
# ============================================================
# CELLULE 19 — Concordance selon la marge T0
# ============================================================

concordance_par_marge = (
    validation_reco[
        validation_reco[
            "PARCOURS_RECOMMANDE"
        ] != "A_VALIDER"
    ]
    .groupby(
        "MARGE_T0",
        as_index=False
    )
    .agg(
        N=("CONCORDANCE_T1", "size"),
        CONCORDANCE_PCT=(
            "CONCORDANCE_T1",
            lambda x: x.mean() * 100
        )
    )
)


display(
    concordance_par_marge.round(2)
)

,MARGE_T0,N,CONCORDANCE_PCT
0,0.0,38,21.05
1,20.0,45,24.44
2,40.0,20,10.00
3,60.0,5,40.00
4,80.0,1,0.00


# Cellule 20 — Matrice recommandation vs parcours T1

In [20]:
# ============================================================
# CELLULE 20 — Matrice descriptive
# ============================================================

matrice_reco_t1 = pd.crosstab(
    validation_decidee[
        "PARCOURS_RECOMMANDE"
    ],
    validation_decidee[
        "T1__PARCOURS"
    ],
    margins=True
)


display(
    matrice_reco_t1
)

T1__PARCOURS,BI,DA,DS,IA,All
PARCOURS_RECOMMANDE,,,,,
BI,6,2,3,5,16
DA,20,6,8,16,50
DS,8,4,3,8,23
IA,3,1,8,8,20
All,37,13,22,37,109


Prochaine cellule : analyser la compatibilité objectif ↔ parcours observé

In [21]:
# ============================================================
# CELLULE 21 — Objectif professionnel vs parcours T1 observé
# ============================================================

analyse_objectif_t1 = (
    BASE_RECO_T0[
        [
            ID,
            "Objectif pro",
            "PARCOURS_OBJECTIF"
        ]
    ]
    .merge(
        BASE_MT1_125[
            [
                ID,
                "T1__PARCOURS"
            ]
        ],
        on=ID,
        how="inner",
        validate="one_to_one"
    )
)


analyse_objectif_t1[
    "OBJECTIF_CONCORDE_T1"
] = (
    analyse_objectif_t1[
        "PARCOURS_OBJECTIF"
    ]
    ==
    analyse_objectif_t1[
        "T1__PARCOURS"
    ]
)


avec_objectif_mappe = (
    analyse_objectif_t1[
        analyse_objectif_t1[
            "PARCOURS_OBJECTIF"
        ].notna()
    ]
)


print(
    "N avec objectif directement mappable :",
    len(avec_objectif_mappe)
)


print(
    "Concordance objectif ↔ T1 :",
    round(
        avec_objectif_mappe[
            "OBJECTIF_CONCORDE_T1"
        ].mean() * 100,
        2
    ),
    "%"
)


matrice_objectif_t1 = pd.crosstab(
    avec_objectif_mappe[
        "PARCOURS_OBJECTIF"
    ],
    avec_objectif_mappe[
        "T1__PARCOURS"
    ],
    margins=True
)


display(
    matrice_objectif_t1
)

N avec objectif directement mappable : 74
Concordance objectif ↔ T1 : 24.32 %


T1__PARCOURS,BI,DA,DS,IA,All
PARCOURS_OBJECTIF,,,,,
BI,7,2,3,2,14
DA,16,3,6,10,35
DS,1,0,1,1,3
IA,4,3,8,7,22
All,28,8,18,20,74


# Cellule 22 — Recommandation multicritère

In [22]:
# ============================================================
# CELLULE 22 — Recommandation multicritère transparente
# ============================================================

PARCOURS = ["DA", "BI", "DS", "IA"]

SCORE_COL = {
    "DA": "S_DA",
    "BI": "S_BI",
    "DS": "S_DS",
    "IA": "S_IA"
}

READY_COL = {
    "DA": "READY_DA",
    "BI": "READY_BI",
    "DS": "READY_DS",
    "IA": "READY_IA"
}


def recommandation_multicritere(row):

    preuves = {
        p: 0
        for p in PARCOURS
    }

    raisons = {
        p: []
        for p in PARCOURS
    }

    # --------------------------------------------------------
    # 1. Critère T0
    # --------------------------------------------------------

    scores_t0 = {
        p: row[SCORE_COL[p]]
        for p in PARCOURS
    }

    max_t0 = max(scores_t0.values())

    meilleurs_t0 = [
        p
        for p in PARCOURS
        if scores_t0[p] == max_t0
    ]

    for p in meilleurs_t0:
        preuves[p] += 1
        raisons[p].append("meilleur_T0")


    # --------------------------------------------------------
    # 2. Critère objectif professionnel
    # --------------------------------------------------------

    objectif = row["PARCOURS_OBJECTIF"]

    if pd.notna(objectif) and objectif in PARCOURS:
        preuves[objectif] += 1
        raisons[objectif].append("objectif_professionnel")


    # --------------------------------------------------------
    # 3. Critère prérequis
    # --------------------------------------------------------

    readiness = {
        p: row[READY_COL[p]]
        for p in PARCOURS
    }

    max_ready = max(readiness.values())

    meilleurs_ready = [
        p
        for p in PARCOURS
        if readiness[p] == max_ready
    ]

    for p in meilleurs_ready:
        preuves[p] += 1
        raisons[p].append("prerequis")


    # --------------------------------------------------------
    # 4. Parcours avec le plus de preuves
    # --------------------------------------------------------

    max_preuves = max(preuves.values())

    candidats = [
        p
        for p in PARCOURS
        if preuves[p] == max_preuves
    ]


    if len(candidats) == 1:
        principal = candidats[0]
    else:
        principal = "A_VALIDER"


    classement = sorted(
        PARCOURS,
        key=lambda p: (
            preuves[p],
            scores_t0[p],
            readiness[p]
        ),
        reverse=True
    )


    return pd.Series({

        "PARCOURS_PRINCIPAL":
            principal,

        "PARCOURS_CANDIDATS":
            " | ".join(candidats),

        "CLASSEMENT_PARCOURS":
            " > ".join(classement),

        "NB_PREUVES_MAX":
            max_preuves,

        "NB_CANDIDATS_MAX":
            len(candidats),

        "PREUVES_DA":
            preuves["DA"],

        "PREUVES_BI":
            preuves["BI"],

        "PREUVES_DS":
            preuves["DS"],

        "PREUVES_IA":
            preuves["IA"],

        "RAISONS_DA":
            " | ".join(raisons["DA"]),

        "RAISONS_BI":
            " | ".join(raisons["BI"]),

        "RAISONS_DS":
            " | ".join(raisons["DS"]),

        "RAISONS_IA":
            " | ".join(raisons["IA"])
    })

# Cellule 23 — Application aux 305 apprenants

In [23]:
# ============================================================
# CELLULE 23 — Application recommandation multicritère
# ============================================================

RECO_MULTI = (
    BASE_RECO_T0
    .apply(
        recommandation_multicritere,
        axis=1
    )
)


BASE_RECO_MULTI = pd.concat(
    [
        BASE_RECO_T0,
        RECO_MULTI
    ],
    axis=1
)


print(
    "N apprenants :",
    len(BASE_RECO_MULTI)
)


display(
    BASE_RECO_MULTI[
        "PARCOURS_PRINCIPAL"
    ]
    .value_counts()
    .rename_axis("PARCOURS_PRINCIPAL")
    .reset_index(name="N")
)

N apprenants : 305


,PARCOURS_PRINCIPAL,N
0,DA,128
1,A_VALIDER,116
2,IA,26
3,DS,23
4,BI,12


# Cellule 24 — Mesurer l’incertitude de recommandation

In [24]:
# ============================================================
# CELLULE 24 — Niveau d'ambiguïté
# ============================================================

resume_preuves = (
    BASE_RECO_MULTI
    .groupby(
        [
            "NB_PREUVES_MAX",
            "NB_CANDIDATS_MAX"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "N"
        }
    )
)


resume_preuves["POURCENTAGE"] = (
    resume_preuves["N"]
    / len(BASE_RECO_MULTI)
    * 100
)


display(
    resume_preuves.round(2)
)

,NB_PREUVES_MAX,NB_CANDIDATS_MAX,N,POURCENTAGE
0,1,2,26,8.52
1,1,3,24,7.87
2,1,4,2,0.66
3,2,1,153,50.16
4,2,2,48,15.74
5,2,3,8,2.62
6,2,4,8,2.62
7,3,1,36,11.80


# Phase C — Niveau Débutant / Intermédiaire / Avancé

# Cellule 25 — Prédiction T2 pour les 125 apprenants

In [25]:
# ============================================================
# CELLULE 25 — Prédiction T2 hors-échantillon
# ============================================================

RF_OOF_125 = (
    OOF_MT1_125[
        OOF_MT1_125[
            "MODELE"
        ] == "RandomForest"
    ]
    .copy()
)


index_ids = (
    BASE_MT1_125[
        [ID]
    ]
    .reset_index()
    .rename(
        columns={
            "index":
                "ROW_INDEX"
        }
    )
)


RF_OOF_125 = (
    RF_OOF_125
    .merge(
        index_ids,
        on="ROW_INDEX",
        how="left",
        validate="one_to_one"
    )
)


assert RF_OOF_125[ID].notna().all()
assert RF_OOF_125[ID].is_unique


BASE_MT1_125 = (
    BASE_MT1_125
    .merge(
        RF_OOF_125[
            [
                ID,
                "Y_PRED",
                "Y_PRED_SD"
            ]
        ],
        on=ID,
        how="left",
        validate="one_to_one"
    )
    .rename(
        columns={
            "Y_PRED":
                "T2_PREDIT_OOF",

            "Y_PRED_SD":
                "T2_PREDIT_OOF_SD"
        }
    )
)


assert (
    BASE_MT1_125[
        "T2_PREDIT_OOF"
    ]
    .notna()
    .all()
)


display(
    BASE_MT1_125[
        [
            ID,
            "T1__PARCOURS",
            "T1__SCORE",
            "T2_PREDIT_OOF",
            "T2_PREDIT_OOF_SD",
            "TARGET_T2"
        ]
    ].head(10)
)

,_ID_NORM,T1__PARCOURS,T1__SCORE,T2_PREDIT_OOF,T2_PREDIT_OOF_SD,TARGET_T2
0,DIEG2026-003,IA,93.333333,77.809068,4.395644,93.333333
1,DIEG2026-007,IA,86.666667,92.789547,0.864486,80.000000
2,DIEG2026-008,BI,33.333333,61.717674,3.785727,100.000000
3,DIEG2026-009,IA,73.333333,66.414313,6.200756,93.333333
4,DIEG2026-012,IA,80.000000,86.425412,2.121904,53.333333
5,DIEG2026-025,DS,46.666667,68.065014,3.538040,86.666667
6,DIEG2026-026,BI,53.333333,68.564324,3.688215,60.000000
7,DIEG2026-027,BI,86.666667,83.625999,2.834552,60.000000
8,DIEG2026-028,BI,60.000000,63.597104,4.254146,80.000000
9,DIEG2026-029,IA,80.000000,77.411420,2.697098,73.333333


# Cellule 26 — Construire 3 niveaux par clustering

In [26]:
# ============================================================
# CELLULE 26 — Vérification des prédictions OOF
# ============================================================

# ------------------------------------------------------------
# 1. Vérifier que la Cellule 25 corrigée a été exécutée
# ------------------------------------------------------------

assert "T2_PREDIT_OOF" in BASE_MT1_125.columns, (
    "La colonne T2_PREDIT_OOF est absente. "
    "Exécuter d'abord la Cellule 25 corrigée."
)


assert (
    BASE_MT1_125[
        "T2_PREDIT_OOF"
    ]
    .notna()
    .all()
), (
    "Certaines prédictions OOF sont manquantes."
)


assert (
    BASE_MT1_125[
        "T2_PREDIT_OOF"
    ]
    .between(
        0,
        100
    )
    .all()
), (
    "Certaines prédictions OOF sont hors de [0, 100]."
)


# ------------------------------------------------------------
# 2. Résumé descriptif
# ------------------------------------------------------------

print(
    "✅ Prédictions OOF disponibles pour :",
    len(BASE_MT1_125),
    "apprenants"
)


display(
    BASE_MT1_125[
        "T2_PREDIT_OOF"
    ]
    .describe()
    .to_frame(
        name="T2_PREDIT_OOF"
    )
)


# ------------------------------------------------------------
# 3. Distribution par parcours
# ------------------------------------------------------------

resume_oof_parcours = (
    BASE_MT1_125

    .groupby(
        "T1__PARCOURS"
    )

    ["T2_PREDIT_OOF"]

    .agg(
        N="count",
        MOYENNE="mean",
        ECART_TYPE="std",
        MEDIANE="median",
        MIN="min",
        MAX="max"
    )

    .reset_index()
)


display(
    resume_oof_parcours.round(3)
)


print(
    "\nℹ️ Aucun niveau pédagogique n'est créé ici."
)

print(
    "➡️ Les segments seront calculés séparément "
    "dans chaque parcours à la Cellule 29."
)

✅ Prédictions OOF disponibles pour : 125 apprenants


,T2_PREDIT_OOF
count,125.000000
mean,71.793942
std,11.679913
min,50.382485
25%,61.717674
50%,70.048420
75%,80.765083
max,94.677680


,T1__PARCOURS,N,MOYENNE,ECART_TYPE,MEDIANE,MIN,MAX
0,BI,42,69.868,10.282,69.287,52.869,88.942
1,DA,17,71.113,12.768,70.868,50.654,92.117
2,DS,26,75.859,12.790,75.219,50.382,94.135
3,IA,40,71.463,11.642,68.728,50.866,94.678



ℹ️ Aucun niveau pédagogique n'est créé ici.
➡️ Les segments seront calculés séparément dans chaque parcours à la Cellule 29.


# Cellule 27 — Caractériser les trois niveaux

In [27]:
# ============================================================
# CELLULE 27 — Ancienne segmentation globale désactivée
# ============================================================

print(
    "ℹ️ Le clustering global des apprenants n'est plus utilisé."
)

print(
    "Les niveaux exploratoires seront calculés séparément "
    "dans chaque parcours DA, BI, DS et IA."
)

print(
    "➡️ Exécuter la Cellule 29 pour construire NIVEAU_PARCOURS "
    "à partir de T2_PREDIT_OOF."
)

ℹ️ Le clustering global des apprenants n'est plus utilisé.
Les niveaux exploratoires seront calculés séparément dans chaque parcours DA, BI, DS et IA.
➡️ Exécuter la Cellule 29 pour construire NIVEAU_PARCOURS à partir de T2_PREDIT_OOF.


# voir comment les 41 Débutants, 54 Intermédiaires et 30 Avancés se répartissent entre les parcours réellement suivis T1.

In [28]:
# ============================================================
# CELLULE 28 — Préparation à la segmentation par parcours
# ============================================================

# La précédente approche utilisait un clustering global :
#
#     T2_PREDIT -> NIVEAU_PREDIT
#
# Cette approche n'est plus retenue.
#
# La nouvelle approche utilise :
#
#     T2_PREDIT_OOF
#          ↓
#     clustering séparé dans chaque parcours
#          ↓
#     NIVEAU_PARCOURS
#
# NIVEAU_PARCOURS sera créé dans la Cellule 29.


# ------------------------------------------------------------
# 1. Vérifier les données nécessaires
# ------------------------------------------------------------

colonnes_requises = {
    "T1__PARCOURS",
    "T2_PREDIT_OOF"
}


colonnes_manquantes = (
    colonnes_requises
    -
    set(BASE_MT1_125.columns)
)


assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes avant la segmentation : "
    f"{colonnes_manquantes}"
)


# ------------------------------------------------------------
# 2. Répartition actuelle des apprenants par parcours
# ------------------------------------------------------------

repartition_parcours = (
    BASE_MT1_125[
        "T1__PARCOURS"
    ]
    .value_counts()
    .reindex(
        [
            "DA",
            "BI",
            "DS",
            "IA"
        ],
        fill_value=0
    )
    .rename_axis(
        "PARCOURS"
    )
    .reset_index(
        name="N_APPRENANTS"
    )
)


display(
    repartition_parcours
)


# ------------------------------------------------------------
# 3. Contrôles
# ------------------------------------------------------------

assert (
    repartition_parcours[
        "N_APPRENANTS"
    ].sum()
    ==
    len(BASE_MT1_125)
)


assert (
    BASE_MT1_125[
        "T2_PREDIT_OOF"
    ]
    .notna()
    .all()
)


print(
    "✅ Données prêtes pour la segmentation par parcours."
)

print(
    "➡️ Exécuter maintenant la Cellule 29 "
    "pour créer NIVEAU_PARCOURS."
)

,PARCOURS,N_APPRENANTS
0,DA,17
1,BI,42
2,DS,26
3,IA,40


✅ Données prêtes pour la segmentation par parcours.
➡️ Exécuter maintenant la Cellule 29 pour créer NIVEAU_PARCOURS.


# Cellule 29 — Clustering séparé dans chaque parcours

In [29]:
# ============================================================
# CELLULE 29 — Segmentation exploratoire OOF par parcours
# ============================================================

from sklearn.cluster import KMeans


BASE_MT1_125[
    "NIVEAU_PARCOURS"
] = pd.NA


BASE_MT1_125[
    "STATUT_NIVEAU"
] = (
    "EXPLORATOIRE_KMEANS_OOF"
)


resume_clusters_parcours = []


for parcours in [
    "DA",
    "BI",
    "DS",
    "IA"
]:

    masque = (
        BASE_MT1_125[
            "T1__PARCOURS"
        ] == parcours
    )


    temp = (
        BASE_MT1_125.loc[
            masque,
            [
                "T2_PREDIT_OOF"
            ]
        ]
        .copy()
    )


    assert len(temp) >= 3


    kmeans = KMeans(
        n_clusters=3,
        random_state=RANDOM_STATE,
        n_init=50
    )


    clusters = (
        kmeans.fit_predict(
            temp[
                ["T2_PREDIT_OOF"]
            ]
        )
    )


    centres = (
        pd.DataFrame({
            "CLUSTER":
                [0, 1, 2],

            "CENTRE":
                kmeans
                .cluster_centers_
                .ravel()
        })

        .sort_values(
            "CENTRE"
        )

        .reset_index(
            drop=True
        )
    )


    mapping = {
        int(
            centres.iloc[0][
                "CLUSTER"
            ]
        ):
            "Debutant",

        int(
            centres.iloc[1][
                "CLUSTER"
            ]
        ):
            "Intermediaire",

        int(
            centres.iloc[2][
                "CLUSTER"
            ]
        ):
            "Avance"
    }


    BASE_MT1_125.loc[
        temp.index,
        "NIVEAU_PARCOURS"
    ] = (
        pd.Series(
            clusters,
            index=temp.index
        )
        .map(mapping)
    )


    for _, ligne in (
        centres.iterrows()
    ):

        resume_clusters_parcours.append({
            "PARCOURS":
                parcours,

            "NIVEAU_EXPLORATOIRE":
                mapping[
                    int(
                        ligne[
                            "CLUSTER"
                        ]
                    )
                ],

            "CENTRE_T2_PREDIT_OOF":
                ligne["CENTRE"]
        })


CENTRES_NIVEAUX = pd.DataFrame(
    resume_clusters_parcours
)


display(
    CENTRES_NIVEAUX
)

,PARCOURS,NIVEAU_EXPLORATOIRE,CENTRE_T2_PREDIT_OOF
0,DA,Debutant,56.473777
1,DA,Intermediaire,69.234239
2,DA,Avance,85.191995
3,BI,Debutant,60.479555
4,BI,Intermediaire,73.277359
5,BI,Avance,83.529208
6,DS,Debutant,64.656434
7,DS,Intermediaire,80.539626
8,DS,Avance,91.137959
9,IA,Debutant,62.313453


In [30]:
# ============================================================
# CELLULE 29A — Répartition Niveau × Parcours
# ============================================================

assert "NIVEAU_PARCOURS" in BASE_MT1_125.columns, (
    "NIVEAU_PARCOURS absent. "
    "Exécuter d'abord la Cellule 29."
)


ordre_niveaux = [
    "Debutant",
    "Intermediaire",
    "Avance"
]


ordre_parcours = [
    "DA",
    "BI",
    "DS",
    "IA"
]


repartition_niveau_parcours = pd.crosstab(
    BASE_MT1_125[
        "T1__PARCOURS"
    ],
    BASE_MT1_125[
        "NIVEAU_PARCOURS"
    ]
)


repartition_niveau_parcours = (
    repartition_niveau_parcours
    .reindex(
        index=ordre_parcours,
        columns=ordre_niveaux,
        fill_value=0
    )
)


repartition_niveau_parcours[
    "TOTAL"
] = (
    repartition_niveau_parcours
    .sum(axis=1)
)


repartition_niveau_parcours.loc[
    "TOTAL"
] = (
    repartition_niveau_parcours
    .sum(axis=0)
)


display(
    repartition_niveau_parcours
)


assert (
    repartition_niveau_parcours.loc[
        "TOTAL",
        "TOTAL"
    ]
    ==
    len(BASE_MT1_125)
)


print(
    "✅ Répartition parcours × niveau exploratoire construite."
)

NIVEAU_PARCOURS,Debutant,Intermediaire,Avance,TOTAL
T1__PARCOURS,,,,
DA,5,6,6,17
BI,20,11,11,42
DS,13,5,8,26
IA,22,10,8,40
TOTAL,60,32,33,125


✅ Répartition parcours × niveau exploratoire construite.


# Cellule 30 — Résultat DA/BI/DS/IA × niveau

In [31]:
# ============================================================
# CELLULE 30 — Répartition finale Parcours × Niveau
# ============================================================

table_niveaux_parcours = pd.crosstab(
    BASE_MT1_125["T1__PARCOURS"],
    BASE_MT1_125["NIVEAU_PARCOURS"]
)


table_niveaux_parcours = (
    table_niveaux_parcours
    .reindex(
        columns=[
            "Debutant",
            "Intermediaire",
            "Avance"
        ],
        fill_value=0
    )
)


table_niveaux_parcours["TOTAL"] = (
    table_niveaux_parcours.sum(axis=1)
)


table_niveaux_parcours.loc["TOTAL"] = (
    table_niveaux_parcours.sum(axis=0)
)


display(
    table_niveaux_parcours
)

NIVEAU_PARCOURS,Debutant,Intermediaire,Avance,TOTAL
T1__PARCOURS,,,,
BI,20,11,11,42
DA,5,6,6,17
DS,13,5,8,26
IA,22,10,8,40
TOTAL,60,32,33,125


# Cellule 31 — Caractériser chaque niveau dans chaque parcours

In [32]:
# ============================================================
# CELLULE 31 — Profil des niveaux par parcours
# ============================================================

resume_niveaux_parcours = (
    BASE_MT1_125

    .groupby(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ],
        as_index=False
    )

    .agg(

        N=(
            ID,
            "size"
        ),

        T2_PREDIT_OOF_MIN=(
            "T2_PREDIT_OOF",
            "min"
        ),

        T2_PREDIT_OOF_MOYEN=(
            "T2_PREDIT_OOF",
            "mean"
        ),

        T2_PREDIT_OOF_MAX=(
            "T2_PREDIT_OOF",
            "max"
        ),

        T1_MOYEN=(
            "T1__SCORE",
            "mean"
        ),

        T2_REEL_MOYEN=(
            "TARGET_T2",
            "mean"
        )
    )
)


display(
    resume_niveaux_parcours.round(2)
)

,T1__PARCOURS,NIVEAU_PARCOURS,N,T2_PREDIT_OOF_MIN,T2_PREDIT_OOF_MOYEN,T2_PREDIT_OOF_MAX,T1_MOYEN,T2_REEL_MOYEN
0,BI,Avance,11,79.06,83.53,88.94,90.30,75.76
1,BI,Debutant,20,52.87,60.48,66.19,50.33,61.00
2,BI,Intermediaire,11,68.56,73.28,77.74,66.67,79.39
3,DA,Avance,6,79.99,85.19,92.12,95.56,86.67
4,DA,Debutant,5,50.65,56.47,61.60,56.00,56.00
5,DA,Intermediaire,6,63.89,69.23,75.26,64.44,83.33
6,DS,Avance,8,86.52,91.14,94.14,94.17,95.83
7,DS,Debutant,13,50.38,64.66,71.17,62.05,63.08
8,DS,Intermediaire,5,79.27,80.54,82.98,84.00,76.00
9,IA,Avance,8,85.05,88.81,94.68,88.33,85.00


Phase D — identifier les lacunes et prérequis
        ↓
DA/BI/DS/IA
        +
Débutant/Intermédiaire/Avancé
        +
lacunes communes
        +
prérequis
        ↓
Phase E — créer les groupes pédagogiques

# Phase D — lacunes et prérequis.

L’idée : ne pas inventer de seuil arbitraire. Une lacune correspondra d’abord à une question T0 incorrecte, donc à une difficulté réellement observée.

Cellule 32 — Charger le scoring détaillé T0

In [33]:
# ============================================================
# CELLULE 32 — Réponses détaillées T1
# ============================================================

T1_LONG = (
    T1_SCORING_LONG
    .copy()
)


print(
    "T1_LONG :",
    T1_LONG.shape
)

print(
    T1_LONG.columns.tolist()
)


assert "CORRECT" in T1_LONG.columns
assert "COMPETENCE" in T1_LONG.columns
assert "PARCOURS" in T1_LONG.columns
assert ID in T1_LONG.columns


display(
    T1_LONG.head()
)

T1_LONG : (2940, 17)
['_ID_NORM', 'ETAPE', 'PARCOURS', 'SOURCE_SHEET', 'QUESTION_INDEX', 'QUESTION', 'REPONSE', 'REPONSE_PRESENTE', 'QUESTION_CORRIGE', 'REPONSE_CORRECTE', 'COMPETENCE', 'REPONSE_BRUTE_MANQUANTE', 'OPTION_REPONSE', 'OPTION_CORRECTE', 'OPTION_NON_RECONNUE', 'CORRECT', 'REPONSE_MANQUANTE']


,_ID_NORM,ETAPE,PARCOURS,SOURCE_SHEET,QUESTION_INDEX,QUESTION,REPONSE,REPONSE_PRESENTE,QUESTION_CORRIGE,REPONSE_CORRECTE,COMPETENCE,REPONSE_BRUTE_MANQUANTE,OPTION_REPONSE,OPTION_CORRECTE,OPTION_NON_RECONNUE,CORRECT,REPONSE_MANQUANTE
0,DIEG2026-003,T1,IA,MI-TEST-IA,1,Pourquoi un LLM hallucine-t-il meme quand on l...,C) Parce qu'il predit le token suivant le plus...,True,Pourquoi un LLM hallucine-t-il meme quand on l...,C,LLM (hallucinations),False,C,C,False,1,False
1,DIEG2026-003,T1,IA,MI-TEST-IA,2,Une information critique est placee au milieu ...,C) Le Lost in the Middle — les LLMs retiennent...,True,Une information critique est placee au milieu ...,C,LLM (contexte long),False,C,C,False,1,False
2,DIEG2026-003,T1,IA,MI-TEST-IA,3,Quelle technique de Prompt Engineering demande...,C) Chain-of-Thought (CoT) — Reflechis etape pa...,True,Quelle technique de Prompt Engineering demande...,C,Prompt engineering,False,C,C,False,1,False
3,DIEG2026-003,T1,IA,MI-TEST-IA,4,Vous construisez un classificateur automatique...,D) temperature = 0 — le modele choisit toujour...,True,Vous construisez un classificateur automatique...,D,LLM (paramètres),False,D,D,False,1,False
4,DIEG2026-003,T1,IA,MI-TEST-IA,5,Pour integrer un LLM dans une application et p...,D) Function Calling / Tool Use — l'API force l...,True,Pour integrer un LLM dans une application et p...,D,LLM (sorties structurées),False,D,D,False,1,False


# Cellule 33 — Vérifier les colonnes nécessaires

In [34]:
# ============================================================
# CELLULE 33 — Vérification des colonnes T1
# ============================================================

# ------------------------------------------------------------
# 1. Vérifier que la Cellule 32 a bien été exécutée
# ------------------------------------------------------------

assert "T1_LONG" in globals(), (
    "T1_LONG n'existe pas. "
    "Exécuter d'abord la Cellule 32."
)


# ------------------------------------------------------------
# 2. Colonnes nécessaires pour l'analyse des lacunes T1
# ------------------------------------------------------------

colonnes_requises = {
    ID,
    "PARCOURS",
    "QUESTION_INDEX",
    "QUESTION",
    "COMPETENCE",
    "CORRECT"
}


colonnes_manquantes = (
    colonnes_requises
    -
    set(T1_LONG.columns)
)


assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes dans T1_LONG : "
    f"{colonnes_manquantes}"
)


# ------------------------------------------------------------
# 3. Afficher les colonnes disponibles
# ------------------------------------------------------------

print(
    "Colonnes disponibles dans T1_LONG :"
)

for i, col in enumerate(
    T1_LONG.columns,
    start=1
):
    print(
        f"{i:02d}. {col}"
    )


# ------------------------------------------------------------
# 4. Contrôles supplémentaires
# ------------------------------------------------------------

print(
    "\nNombre de lignes :",
    len(T1_LONG)
)

print(
    "Nombre d'apprenants :",
    T1_LONG[
        ID
    ].nunique()
)

print(
    "Parcours observés :",
    sorted(
        T1_LONG[
            "PARCOURS"
        ]
        .dropna()
        .unique()
        .tolist()
    )
)


# ------------------------------------------------------------
# 5. Vérifier les valeurs de CORRECT
# ------------------------------------------------------------

print(
    "\nValeurs observées dans CORRECT :"
)

display(
    T1_LONG[
        "CORRECT"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "N"
    )
    .to_frame()
)


print(
    "\n✅ T1_LONG prêt pour l'analyse des lacunes."
)

Colonnes disponibles dans T1_LONG :
01. _ID_NORM
02. ETAPE
03. PARCOURS
04. SOURCE_SHEET
05. QUESTION_INDEX
06. QUESTION
07. REPONSE
08. REPONSE_PRESENTE
09. QUESTION_CORRIGE
10. REPONSE_CORRECTE
11. COMPETENCE
12. REPONSE_BRUTE_MANQUANTE
13. OPTION_REPONSE
14. OPTION_CORRECTE
15. OPTION_NON_RECONNUE
16. CORRECT
17. REPONSE_MANQUANTE

Nombre de lignes : 2940
Nombre d'apprenants : 196
Parcours observés : ['BI', 'DA', 'DS', 'IA']

Valeurs observées dans CORRECT :


,N
CORRECT,
1,2119
0,821



✅ T1_LONG prêt pour l'analyse des lacunes.


# Cellule 34 — Construire les lacunes T0 par apprenant et domaine

In [35]:
# ============================================================
# CELLULE 34 — Lacunes observées à T1
# ============================================================

LACUNES_T1_LONG = (
    T1_LONG[
        T1_LONG[
            "CORRECT"
        ].eq(0)
        &
        T1_LONG[
            ID
        ].isin(
            BASE_MT1_125[
                ID
            ]
        )
    ]
    .copy()
)


lacunes_par_apprenant = (

    LACUNES_T1_LONG

    .groupby(
        [
            ID,
            "PARCOURS"
        ],
        as_index=False
    )

    .agg(
        NB_LACUNES=(
            "QUESTION_INDEX",
            "size"
        ),

        QUESTIONS_LACUNES=(
            "QUESTION",
            lambda x:
                " | ".join(
                    sorted(
                        x
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                )
        ),

        COMPETENCES_LACUNES=(
            "COMPETENCE",
            lambda x:
                " | ".join(
                    sorted(
                        x
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                )
        )
    )
)


display(
    lacunes_par_apprenant.head()
)

,_ID_NORM,PARCOURS,NB_LACUNES,QUESTIONS_LACUNES,COMPETENCES_LACUNES
0,DIEG2026-003,IA,1,"Dans Open WebUI, quelle est la difference fond...",Open WebUI (Skills)
1,DIEG2026-007,IA,2,"Selon la regle des 50 %, quelle information de...",LLM (paramètres) | Open WebUI (règle des 50 %)
2,DIEG2026-008,BI,10,Comment resoudre correctement une relation plu...,Modèle en étoile | Nettoyage de cellules fusio...
3,DIEG2026-009,IA,4,Parmi les 6 composantes d'un systeme prompt pr...,LLM (sorties structurées) | Open WebUI (règle ...
4,DIEG2026-012,IA,3,"Dans Open WebUI, quelle est la difference fond...",LLM (contexte long) | LLM (paramètres) | Open ...


# Phase E — groupes pédagogiques.

# Cellule 35 — Ajouter les lacunes au parcours suivi

In [36]:
# ============================================================
# CELLULE 35 — Fusion des lacunes T1 avec N=125
# ============================================================

lacunes_parcours = (
    lacunes_par_apprenant
    .rename(
        columns={
            "PARCOURS":
                "T1__PARCOURS"
        }
    )
)


BASE_GROUPES = (
    BASE_MT1_125
    .merge(
        lacunes_parcours,
        on=[
            ID,
            "T1__PARCOURS"
        ],
        how="left",
        validate="one_to_one"
    )
)


BASE_GROUPES[
    "NB_LACUNES"
] = (
    BASE_GROUPES[
        "NB_LACUNES"
    ]
    .fillna(0)
    .astype(int)
)


for col in [
    "QUESTIONS_LACUNES",
    "COMPETENCES_LACUNES"
]:

    BASE_GROUPES[
        col
    ] = (
        BASE_GROUPES[col]
        .fillna(
            "Aucune lacune T1 détectée"
        )
    )


# Objectif professionnel explicite
if (
    "Objectif pro"
    not in BASE_GROUPES.columns
):

    BASE_GROUPES = (
        BASE_GROUPES
        .merge(
            D1[
                [
                    ID,
                    "Objectif pro"
                ]
            ],
            on=ID,
            how="left",
            validate="one_to_one"
        )
    )


display(
    BASE_GROUPES[
        [
            ID,
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "NB_LACUNES",
            "COMPETENCES_LACUNES"
        ]
    ].head()
)

,_ID_NORM,T1__PARCOURS,NIVEAU_PARCOURS,NB_LACUNES,COMPETENCES_LACUNES
0,DIEG2026-003,IA,Intermediaire,1,Open WebUI (Skills)
1,DIEG2026-007,IA,Avance,2,LLM (paramètres) | Open WebUI (règle des 50 %)
2,DIEG2026-008,BI,Debutant,10,Modèle en étoile | Nettoyage de cellules fusio...
3,DIEG2026-009,IA,Debutant,4,LLM (sorties structurées) | Open WebUI (règle ...
4,DIEG2026-012,IA,Avance,3,LLM (contexte long) | LLM (paramètres) | Open ...


# Cellule 36 — Préparer les prérequis par parcours

In [37]:
# ============================================================
# CELLULE 36 — Profil de prérequis
# ============================================================

def profil_prerequis(row):

    parcours = row["T1__PARCOURS"]

    if parcours == "DA":
        return (
            f"INFO={row['P__NIVEAU_INFORMATIQUE']} | "
            f"EXCEL={row['P__NIVEAU_EXCEL']}"
        )

    if parcours == "BI":
        return (
            f"INFO={row['P__NIVEAU_INFORMATIQUE']} | "
            f"EXCEL={row['P__NIVEAU_EXCEL']} | "
            f"POWERBI={row['P__NIVEAU_POWER_BI']}"
        )

    if parcours == "DS":
        return (
            f"INFO={row['P__NIVEAU_INFORMATIQUE']} | "
            f"PYTHON={row['P__NIVEAU_PYTHON']}"
        )

    if parcours == "IA":
        return (
            f"INFO={row['P__NIVEAU_INFORMATIQUE']} | "
            f"PYTHON={row['P__NIVEAU_PYTHON']} | "
            f"IA={row['P__NIVEAU_IA']}"
        )

    return "NON_DEFINI"


BASE_GROUPES[
    "PROFIL_PREREQUIS"
] = BASE_GROUPES.apply(
    profil_prerequis,
    axis=1
)


print("✅ Prérequis ajoutés")

✅ Prérequis ajoutés


# Cellule 37 — Résumé Parcours × Niveau × Lacunes

In [38]:
# ============================================================
# CELLULE 37 — Résumé Phase D
# ============================================================

# ------------------------------------------------------------
# 1. Vérifications
# ------------------------------------------------------------

colonnes_requises = {
    ID,
    "T1__PARCOURS",
    "NIVEAU_PARCOURS",
    "NB_LACUNES",
    "T1__SCORE",
    "T2_PREDIT_OOF"
}


colonnes_manquantes = (
    colonnes_requises
    -
    set(BASE_GROUPES.columns)
)


assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes dans BASE_GROUPES : "
    f"{colonnes_manquantes}"
)


# ------------------------------------------------------------
# 2. Résumé par parcours et niveau exploratoire
# ------------------------------------------------------------

resume_phase_D = (

    BASE_GROUPES

    .groupby(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ],
        as_index=False,
        observed=True
    )

    .agg(

        N=(
            ID,
            "size"
        ),

        NB_LACUNES_MOYEN=(
            "NB_LACUNES",
            "mean"
        ),

        NB_LACUNES_MEDIAN=(
            "NB_LACUNES",
            "median"
        ),

        T1_MOYEN=(
            "T1__SCORE",
            "mean"
        ),

        T2_PREDIT_OOF_MOYEN=(
            "T2_PREDIT_OOF",
            "mean"
        ),

        T2_PREDIT_OOF_STD=(
            "T2_PREDIT_OOF",
            "std"
        ),

        T2_REEL_MOYEN=(
            "TARGET_T2",
            "mean"
        )
    )
)


# ------------------------------------------------------------
# 3. Ordonner les niveaux
# ------------------------------------------------------------

ordre_niveaux = [
    "Debutant",
    "Intermediaire",
    "Avance"
]


ordre_parcours = [
    "DA",
    "BI",
    "DS",
    "IA"
]


resume_phase_D[
    "NIVEAU_PARCOURS"
] = pd.Categorical(
    resume_phase_D[
        "NIVEAU_PARCOURS"
    ],
    categories=ordre_niveaux,
    ordered=True
)


resume_phase_D[
    "T1__PARCOURS"
] = pd.Categorical(
    resume_phase_D[
        "T1__PARCOURS"
    ],
    categories=ordre_parcours,
    ordered=True
)


resume_phase_D = (
    resume_phase_D
    .sort_values(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 4. Affichage
# ------------------------------------------------------------

display(
    resume_phase_D.round(2)
)


# ------------------------------------------------------------
# 5. Contrôles
# ------------------------------------------------------------

assert (
    resume_phase_D[
        "N"
    ].sum()
    ==
    len(BASE_GROUPES)
)


print(
    "✅ Résumé Phase D construit avec les prédictions OOF."
)

,T1__PARCOURS,NIVEAU_PARCOURS,N,NB_LACUNES_MOYEN,NB_LACUNES_MEDIAN,T1_MOYEN,T2_PREDIT_OOF_MOYEN,T2_PREDIT_OOF_STD,T2_REEL_MOYEN
0,DA,Debutant,5,6.60,7.0,56.00,56.47,4.79,56.00
1,DA,Intermediaire,6,5.33,6.0,64.44,69.23,4.42,83.33
2,DA,Avance,6,0.67,0.0,95.56,85.19,5.24,86.67
3,BI,Debutant,20,7.45,7.0,50.33,60.48,3.30,61.00
4,BI,Intermediaire,11,5.00,3.0,66.67,73.28,3.03,79.39
5,BI,Avance,11,1.45,2.0,90.30,83.53,3.02,75.76
6,DS,Debutant,13,5.69,5.0,62.05,64.66,6.05,63.08
7,DS,Intermediaire,5,2.40,3.0,84.00,80.54,1.50,76.00
8,DS,Avance,8,0.88,0.5,94.17,91.14,2.19,95.83
9,IA,Debutant,22,6.86,6.0,54.24,62.31,4.93,60.91


✅ Résumé Phase D construit avec les prédictions OOF.


Après cette cellule, chaque apprenant possède bien :
Parcours
+ Niveau dans le parcours
+ Lacunes observées
+ Compétences en lacune
+ Prérequis

Phase E — création des groupes pédagogiques


# Cellule 38 — Niveau de prérequis du parcours
Groupe =Parcours+ Niveau+ Lacunes communes+ Prérequis compatibles

In [39]:
# ============================================================
# CELLULE 38 — Niveau de prérequis limitant
# ============================================================

NIVEAU_LABEL = {
    0: "Aucun",
    1: "Debutant",
    2: "Intermediaire",
    3: "Avance"
}


def prerequis_limitant(row):

    parcours = row["T1__PARCOURS"]

    if parcours == "DA":
        valeurs = [
            row["P__NIVEAU_INFORMATIQUE"],
            row["P__NIVEAU_EXCEL"]
        ]

    elif parcours == "BI":
        valeurs = [
            row["P__NIVEAU_INFORMATIQUE"],
            row["P__NIVEAU_EXCEL"],
            row["P__NIVEAU_POWER_BI"]
        ]

    elif parcours == "DS":
        valeurs = [
            row["P__NIVEAU_INFORMATIQUE"],
            row["P__NIVEAU_PYTHON"]
        ]

    elif parcours == "IA":
        valeurs = [
            row["P__NIVEAU_INFORMATIQUE"],
            row["P__NIVEAU_PYTHON"],
            row["P__NIVEAU_IA"]
        ]

    else:
        return pd.NA

    valeurs = [
        v
        for v in valeurs
        if pd.notna(v)
    ]

    if len(valeurs) == 0:
        return pd.NA

    return NIVEAU_LABEL[
        int(min(valeurs))
    ]


BASE_GROUPES[
    "PREREQUIS_LIMITANT"
] = BASE_GROUPES.apply(
    prerequis_limitant,
    axis=1
)


display(
    BASE_GROUPES[
        [
            ID,
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "PREREQUIS_LIMITANT"
        ]
    ].head(10)
)

,_ID_NORM,T1__PARCOURS,NIVEAU_PARCOURS,PREREQUIS_LIMITANT
0,DIEG2026-003,IA,Intermediaire,Aucun
1,DIEG2026-007,IA,Avance,Debutant
2,DIEG2026-008,BI,Debutant,Debutant
3,DIEG2026-009,IA,Debutant,Debutant
4,DIEG2026-012,IA,Avance,Debutant
5,DIEG2026-025,DS,Debutant,Debutant
6,DIEG2026-026,BI,Intermediaire,Aucun
7,DIEG2026-027,BI,Avance,Aucun
8,DIEG2026-028,BI,Debutant,Debutant
9,DIEG2026-029,IA,Intermediaire,Aucun


# Cellule 39 — Transformer les compétences en lacune en format long

In [40]:
# ============================================================
# CELLULE 39 — Lacunes en format long
# ============================================================

lacunes_competences_long = (
    LACUNES_T1_LONG[
        [
            ID,
            "PARCOURS",
            "COMPETENCE"
        ]
    ]
    .dropna(
        subset=["COMPETENCE"]
    )
    .drop_duplicates()
)


# Ne conserver que la lacune du parcours effectivement suivi
lacunes_groupes_long = (
    BASE_GROUPES[
        [
            ID,
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ]
    ]
    .merge(
        lacunes_competences_long,
        left_on=[
            ID,
            "T1__PARCOURS"
        ],
        right_on=[
            ID,
            "PARCOURS"
        ],
        how="left"
    )
)


display(
    lacunes_groupes_long.head(20)
)

,_ID_NORM,T1__PARCOURS,NIVEAU_PARCOURS,PARCOURS,COMPETENCE
0,DIEG2026-003,IA,Intermediaire,IA,Open WebUI (Skills)
1,DIEG2026-007,IA,Avance,IA,LLM (paramètres)
2,DIEG2026-007,IA,Avance,IA,Open WebUI (règle des 50 %)
3,DIEG2026-008,BI,Debutant,BI,Écosystème Power BI
4,DIEG2026-008,BI,Debutant,BI,Rôle de l'analyste BI
5,DIEG2026-008,BI,Debutant,BI,Power Query (ETL)
6,DIEG2026-008,BI,Debutant,BI,Power Query (Fill Down)
7,DIEG2026-008,BI,Debutant,BI,Power Query (Unpivot)
8,DIEG2026-008,BI,Debutant,BI,Typage des clés
9,DIEG2026-008,BI,Debutant,BI,Nettoyage de cellules fusionnées


# Cellule 40 — Fréquence des lacunes par parcours et niveau

In [41]:
# ============================================================
# CELLULE 40 — Lacunes communes
# ============================================================

frequence_lacunes = (
    lacunes_groupes_long

    .dropna(
        subset=["COMPETENCE"]
    )

    .groupby(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "COMPETENCE"
        ],
        as_index=False
    )

    .agg(
        N_APPRENANTS=(
            ID,
            "nunique"
        )
    )

    .sort_values(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "N_APPRENANTS"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)


display(
    frequence_lacunes.head(30)
)

,T1__PARCOURS,NIVEAU_PARCOURS,COMPETENCE,N_APPRENANTS
1,BI,Avance,Modèle en étoile,3
6,BI,Avance,Relations Power BI,3
0,BI,Avance,Cardinalité,2
2,BI,Avance,Nettoyage de cellules fusionnées,2
3,BI,Avance,Niveaux d'analyse,2
5,BI,Avance,Power Query (Fill Down),2
4,BI,Avance,Power Query (ETL),1
7,BI,Avance,Rôle de l'analyste BI,1
21,BI,Debutant,Typage des clés,16
13,BI,Debutant,Nettoyage de cellules fusionnées,15


# Cellule 41 — Déterminer la lacune principale de chaque apprenant

Pour éviter de créer un groupe différent pour chaque combinaison de questions, on attribue à chaque apprenant sa lacune la plus commune parmi les apprenants du même parcours et niveau.

In [42]:
# ============================================================
# CELLULE 41 — Lacune principale
# ============================================================

lacunes_avec_frequence = (
    lacunes_groupes_long

    .merge(
        frequence_lacunes,
        on=[
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "COMPETENCE"
        ],
        how="left"
    )
)


lacune_principale = (
    lacunes_avec_frequence

    .dropna(
        subset=["COMPETENCE"]
    )

    .sort_values(
        [
            ID,
            "N_APPRENANTS",
            "COMPETENCE"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )

    .drop_duplicates(
        subset=[ID],
        keep="first"
    )

    [
        [
            ID,
            "COMPETENCE"
        ]
    ]

    .rename(
        columns={
            "COMPETENCE":
                "LACUNE_PRINCIPALE"
        }
    )
)


BASE_GROUPES = (
    BASE_GROUPES

    .merge(
        lacune_principale,
        on=ID,
        how="left",
        validate="one_to_one"
    )
)


BASE_GROUPES[
    "LACUNE_PRINCIPALE"
] = (
    BASE_GROUPES[
        "LACUNE_PRINCIPALE"
    ]
    .fillna(
        "Aucune_lacune_T1"
    )
)


display(
    BASE_GROUPES[
        [
            ID,
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "LACUNE_PRINCIPALE"
        ]
    ].head(20)
)

,_ID_NORM,T1__PARCOURS,NIVEAU_PARCOURS,LACUNE_PRINCIPALE
0,DIEG2026-003,IA,Intermediaire,Open WebUI (Skills)
1,DIEG2026-007,IA,Avance,LLM (paramètres)
2,DIEG2026-008,BI,Debutant,Typage des clés
3,DIEG2026-009,IA,Debutant,Open WebUI (règle des 50 %)
4,DIEG2026-012,IA,Avance,Open WebUI (Skills)
5,DIEG2026-025,DS,Debutant,Machine Learning (généralisation)
6,DIEG2026-026,BI,Intermediaire,Typage des clés
7,DIEG2026-027,BI,Avance,Niveaux d'analyse
8,DIEG2026-028,BI,Debutant,Nettoyage de cellules fusionnées
9,DIEG2026-029,IA,Intermediaire,Open WebUI (Skills)


# Cellule 42 — Créer la signature pédagogique

In [43]:
# ============================================================
# CELLULE 42 — Signature collective
# ============================================================

BASE_GROUPES[
    "SIGNATURE_GROUPE"
] = (
    BASE_GROUPES[
        "T1__PARCOURS"
    ].astype(str)

    + " | "

    + BASE_GROUPES[
        "NIVEAU_PARCOURS"
    ].astype(str)
)


print(
    "Nombre de groupes macro :",
    BASE_GROUPES[
        "SIGNATURE_GROUPE"
    ].nunique()
)

Nombre de groupes macro : 12


# Cellule 43 — Créer les groupes pédagogiques

In [44]:
# ============================================================
# CELLULE 43 — Groupes pédagogiques collectifs
# ============================================================

def top_modalites(
    serie,
    n=3
):

    s = (
        serie
        .dropna()
        .astype(str)
    )

    s = s[
        ~s.str.startswith(
            "Aucune"
        )
    ]

    if len(s) == 0:
        return "Aucune"

    return " | ".join(
        s.value_counts()
        .head(n)
        .index
        .tolist()
    )


def modalite_dominante(
    serie
):

    s = (
        serie
        .dropna()
        .astype(str)
    )

    if len(s) == 0:
        return pd.NA

    return (
        s.value_counts()
        .index[0]
    )


groupes_pedagogiques = (

    BASE_GROUPES

    .groupby(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ],
        as_index=False,
        dropna=False
    )

    .agg(
        N_APPRENANTS=(
            ID,
            "nunique"
        ),

        T1_MOYEN=(
            "T1__SCORE",
            "mean"
        ),

        T2_PREDIT_OOF_MOYEN=(
            "T2_PREDIT_OOF",
            "mean"
        ),

        NB_LACUNES_MOYEN=(
            "NB_LACUNES",
            "mean"
        ),

        LACUNES_PRIORITAIRES=(
            "LACUNE_PRINCIPALE",
            top_modalites
        ),

        PREREQUIS_DOMINANT=(
            "PREREQUIS_LIMITANT",
            modalite_dominante
        ),

        OBJECTIFS_DOMINANTS=(
            "Objectif pro",
            lambda x:
                top_modalites(
                    x,
                    n=2
                )
        )
    )
)


groupes_pedagogiques = (
    groupes_pedagogiques
    .sort_values(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ]
    )
    .reset_index(drop=True)
)


groupes_pedagogiques[
    "ID_GROUPE"
] = [
    f"G{n:03d}"
    for n in range(
        1,
        len(groupes_pedagogiques) + 1
    )
]


display(
    groupes_pedagogiques
)

,T1__PARCOURS,NIVEAU_PARCOURS,N_APPRENANTS,T1_MOYEN,T2_PREDIT_OOF_MOYEN,NB_LACUNES_MOYEN,LACUNES_PRIORITAIRES,PREREQUIS_DOMINANT,OBJECTIFS_DOMINANTS,ID_GROUPE
0,BI,Avance,11,90.303030,83.529208,1.454545,Modèle en étoile | Relations Power BI | Cardin...,Aucun,Data Analyst | Business Analyst,G001
1,BI,Debutant,20,50.333333,60.479555,7.450000,Typage des clés | Nettoyage de cellules fusion...,Aucun,Data Analyst | Entrepreneur,G002
2,BI,Intermediaire,11,66.666667,73.277359,5.000000,Typage des clés | Relations plusieurs-à-plusie...,Aucun,Entrepreneur | Data Analyst,G003
3,DA,Avance,6,95.555556,85.191995,0.666667,Power Query (évaluation paresseuse) | Power Pi...,Debutant,Entrepreneur | Developpeur Python / IA,G004
4,DA,Debutant,5,56.000000,56.473777,6.600000,Power Query (fusion),Debutant,Data Analyst | Non defini,G005
5,DA,Intermediaire,6,64.444444,69.234239,5.333333,Power Pivot (schéma en étoile) | Power Query (...,Debutant,Entrepreneur | Business Analyst,G006
6,DS,Avance,8,94.166667,91.137959,0.875000,Machine Learning (déséquilibre) | Pandas (join...,Debutant,Developpeur Python / IA | Data Analyst,G007
7,DS,Debutant,13,62.051282,64.656434,5.692308,Machine Learning (généralisation) | Pandas (jo...,Debutant,Data Analyst | Entrepreneur,G008
8,DS,Intermediaire,5,84.000000,80.539626,2.400000,Machine Learning (déséquilibre) | Statistiques...,Intermediaire,Data Analyst | Entrepreneur,G009
9,IA,Avance,8,88.333333,88.810156,1.750000,Open WebUI (Skills) | LLM (paramètres) | Open ...,Debutant,Developpeur Python / IA | Entrepreneur,G010


# Cellule 44 — Vérifier si les groupes sont réellement collectifs

In [45]:
# ============================================================
# CELLULE 44 — Taille des groupes
# ============================================================

groupes_pedagogiques[
    "STATUT_GROUPE"
] = np.where(
    groupes_pedagogiques[
        "N_APPRENANTS"
    ] >= 2,
    "GROUPE_COLLECTIF",
    "A_VALIDER"
)


display(
    groupes_pedagogiques[
        [
            "ID_GROUPE",
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "N_APPRENANTS",
            "STATUT_GROUPE"
        ]
    ]
)


print(
    "Singletons :",
    (
        groupes_pedagogiques[
            "N_APPRENANTS"
        ] == 1
    ).sum()
)

,ID_GROUPE,T1__PARCOURS,NIVEAU_PARCOURS,N_APPRENANTS,STATUT_GROUPE
0,G001,BI,Avance,11,GROUPE_COLLECTIF
1,G002,BI,Debutant,20,GROUPE_COLLECTIF
2,G003,BI,Intermediaire,11,GROUPE_COLLECTIF
3,G004,DA,Avance,6,GROUPE_COLLECTIF
4,G005,DA,Debutant,5,GROUPE_COLLECTIF
5,G006,DA,Intermediaire,6,GROUPE_COLLECTIF
6,G007,DS,Avance,8,GROUPE_COLLECTIF
7,G008,DS,Debutant,13,GROUPE_COLLECTIF
8,G009,DS,Intermediaire,5,GROUPE_COLLECTIF
9,G010,IA,Avance,8,GROUPE_COLLECTIF


Singletons : 0


# Cellule 45 — Résumé par parcours et niveau

In [46]:
# ============================================================
# CELLULE 45 — Résumé des groupes
# ============================================================

# ------------------------------------------------------------
# 1. Vérifications
# ------------------------------------------------------------

colonnes_requises = {
    ID,
    "T1__PARCOURS",
    "NIVEAU_PARCOURS",
    "NB_LACUNES",
    "T1__SCORE",
    "T2_PREDIT_OOF"
}

colonnes_manquantes = (
    colonnes_requises
    -
    set(BASE_GROUPES.columns)
)

assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes dans BASE_GROUPES : "
    f"{colonnes_manquantes}"
)


# ------------------------------------------------------------
# 2. Résumé des groupes
# ------------------------------------------------------------

resume_groupes = (

    BASE_GROUPES

    .groupby(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ],
        as_index=False,
        observed=True
    )

    .agg(

        N_APPRENANTS=(
            ID,
            "nunique"
        ),

        NB_LACUNES_MOYEN=(
            "NB_LACUNES",
            "mean"
        ),

        NB_LACUNES_MEDIAN=(
            "NB_LACUNES",
            "median"
        ),

        T1_MOYEN=(
            "T1__SCORE",
            "mean"
        ),

        T2_PREDIT_OOF_MOYEN=(
            "T2_PREDIT_OOF",
            "mean"
        ),

        T2_PREDIT_OOF_STD=(
            "T2_PREDIT_OOF",
            "std"
        )
    )
)


# ------------------------------------------------------------
# 3. Ordonner les parcours et niveaux
# ------------------------------------------------------------

ordre_parcours = [
    "DA",
    "BI",
    "DS",
    "IA"
]

ordre_niveaux = [
    "Debutant",
    "Intermediaire",
    "Avance"
]


resume_groupes[
    "T1__PARCOURS"
] = pd.Categorical(
    resume_groupes[
        "T1__PARCOURS"
    ],
    categories=ordre_parcours,
    ordered=True
)


resume_groupes[
    "NIVEAU_PARCOURS"
] = pd.Categorical(
    resume_groupes[
        "NIVEAU_PARCOURS"
    ],
    categories=ordre_niveaux,
    ordered=True
)


resume_groupes = (
    resume_groupes
    .sort_values(
        [
            "T1__PARCOURS",
            "NIVEAU_PARCOURS"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 4. Affichage
# ------------------------------------------------------------

display(
    resume_groupes.round(2)
)


# ------------------------------------------------------------
# 5. Contrôles
# ------------------------------------------------------------

assert (
    resume_groupes[
        "N_APPRENANTS"
    ].sum()
    ==
    BASE_GROUPES[
        ID
    ].nunique()
)


print(
    "✅ Résumé des groupes construit avec T2_PREDIT_OOF."
)

,T1__PARCOURS,NIVEAU_PARCOURS,N_APPRENANTS,NB_LACUNES_MOYEN,NB_LACUNES_MEDIAN,T1_MOYEN,T2_PREDIT_OOF_MOYEN,T2_PREDIT_OOF_STD
0,DA,Debutant,5,6.60,7.0,56.00,56.47,4.79
1,DA,Intermediaire,6,5.33,6.0,64.44,69.23,4.42
2,DA,Avance,6,0.67,0.0,95.56,85.19,5.24
3,BI,Debutant,20,7.45,7.0,50.33,60.48,3.30
4,BI,Intermediaire,11,5.00,3.0,66.67,73.28,3.03
5,BI,Avance,11,1.45,2.0,90.30,83.53,3.02
6,DS,Debutant,13,5.69,5.0,62.05,64.66,6.05
7,DS,Intermediaire,5,2.40,3.0,84.00,80.54,1.50
8,DS,Avance,8,0.88,0.5,94.17,91.14,2.19
9,IA,Debutant,22,6.86,6.0,54.24,62.31,4.93


✅ Résumé des groupes construit avec T2_PREDIT_OOF.


In [47]:
# ============================================================
# CELLULE 46 — Validation finale des groupes
# ============================================================

groupes_pedagogiques["STATUT_GROUPE"] = np.where(
    groupes_pedagogiques["N_APPRENANTS"] >= 2,
    "GROUPE_COLLECTIF",
    "A_REGROUPER"
)

resume_validation_groupes = (
    groupes_pedagogiques["STATUT_GROUPE"]
    .value_counts()
    .rename_axis("STATUT_GROUPE")
    .reset_index(name="N_GROUPES")
)

display(resume_validation_groupes)

print(
    "Apprenants dans groupes collectifs :",
    groupes_pedagogiques.loc[
        groupes_pedagogiques["N_APPRENANTS"] >= 2,
        "N_APPRENANTS"
    ].sum()
)

print(
    "Apprenants à regrouper :",
    groupes_pedagogiques.loc[
        groupes_pedagogiques["N_APPRENANTS"] == 1,
        "N_APPRENANTS"
    ].sum()
)

,STATUT_GROUPE,N_GROUPES
0,GROUPE_COLLECTIF,12


Apprenants dans groupes collectifs : 125
Apprenants à regrouper : 0


avec 88,8 % des apprenants déjà regroupés collectivement.

# Phase F — recommandation de contenus pédagogiques par groupe

Chaque parcours possède un programme commun
DA → même programme DA pour tous
BI → même programme BI pour tous
DS → même programme DS pour tous
IA → même programme IA pour tous

À l’intérieur d’un parcours, tous suivent donc les mêmes chapitres / leçons / programme principal.
La personnalisation intervient seulement selon le groupe :

Parcours+Niveau+Lacunes+Prérequis===Adaptation pédagogique

example:
Parcours DA
Programme commun :
Chapitre 1
Chapitre 2
Chapitre 3
Chapitre 4
...

Groupe DA-Débutant
→ même programme DA
→ davantage d'explications
→ exercices guidés
→ remédiation sur les lacunes détectées

Groupe DA-Intermédiaire
→ même programme DA
→ exercices ciblés
→ rappels sur les lacunes

Groupe DA-Avancé
→ même programme DA
→ exercices plus complexes
→ approfondissement
→ correction des lacunes spécifiques


Programme = Parcours

et

Personnalisation =Niveau + Lacunes + Prérequis


Donc on ne recommande pas un programme différent par groupe.


# Cellule 47 — Rattacher chaque apprenant à son groupe
Contenu =f(Parcours,\ Niveau,\ Lacunes,\ Prérequis,\ Objectifs)


In [48]:
# ============================================================
# CELLULE 47 — Affectation collective
# ============================================================

CLES_GROUPE = [
    "T1__PARCOURS",
    "NIVEAU_PARCOURS"
]


BASE_GROUPES = (
    BASE_GROUPES
    .merge(
        groupes_pedagogiques[
            CLES_GROUPE
            + [
                "ID_GROUPE",
                "N_APPRENANTS",
                "STATUT_GROUPE",
                "LACUNES_PRIORITAIRES",
                "PREREQUIS_DOMINANT",
                "OBJECTIFS_DOMINANTS"
            ]
        ],
        on=CLES_GROUPE,
        how="left",
        validate="many_to_one"
    )
)


assert (
    BASE_GROUPES[
        "ID_GROUPE"
    ]
    .notna()
    .all()
)


print(
    "Apprenants affectés :",
    len(BASE_GROUPES)
)

print(
    "Groupes :",
    BASE_GROUPES[
        "ID_GROUPE"
    ].nunique()
)

Apprenants affectés : 125
Groupes : 12


# Cellule 48 — Programme commun par parcours

In [49]:
# ============================================================
# CELLULE 48 — Programme commun du parcours
# ============================================================

PROGRAMME_PARCOURS = {
    "DA": "PROGRAMME_DA",
    "BI": "PROGRAMME_BI",
    "DS": "PROGRAMME_DS",
    "IA": "PROGRAMME_IA"
}


BASE_GROUPES["PROGRAMME_COMMUN"] = (
    BASE_GROUPES["T1__PARCOURS"]
    .map(PROGRAMME_PARCOURS)
)


print("✅ Programme commun affecté selon le parcours")

✅ Programme commun affecté selon le parcours


# Cellule 49 — Adaptation selon le niveau
Le programme ne change pas. Seule la forme pédagogique change.

In [50]:
# ============================================================
# CELLULE 49 — Adaptation pédagogique selon le niveau
# ============================================================

ADAPTATION_NIVEAU = {

    "Debutant": {
        "TYPE_ACCOMPAGNEMENT":
            "Renforce",
        "EXPLICATION":
            "Detaillee et progressive",
        "EXERCICES":
            "Guides puis progressifs"
    },

    "Intermediaire": {
        "TYPE_ACCOMPAGNEMENT":
            "Standard cible",
        "EXPLICATION":
            "Rappels et consolidation",
        "EXERCICES":
            "Application et cas pratiques"
    },

    "Avance": {
        "TYPE_ACCOMPAGNEMENT":
            "Approfondissement",
        "EXPLICATION":
            "Synthetique avec approfondissements",
        "EXERCICES":
            "Complexes, challenges et mini-projets"
    }
}


adaptations = (
    BASE_GROUPES["NIVEAU_PARCOURS"]
    .apply(
        lambda niveau:
            pd.Series(
                ADAPTATION_NIVEAU[niveau]
            )
    )
)


BASE_GROUPES = pd.concat(
    [
        BASE_GROUPES,
        adaptations
    ],
    axis=1
)

BASE_GROUPES[
    "STATUT_ADAPTATION"
] = (
    "REGLE_PEDAGOGIQUE_HEURISTIQUE_PROTO"
)

# Cellule 50 — Remédiation spécifique aux lacunes

In [51]:
# ============================================================
# CELLULE 50 — Remédiation selon les lacunes du groupe
# ============================================================

# ------------------------------------------------------------
# 1. Vérifications
# ------------------------------------------------------------

colonnes_requises = {
    "ID_GROUPE",
    "LACUNES_PRIORITAIRES"
}

colonnes_manquantes = (
    colonnes_requises
    -
    set(BASE_GROUPES.columns)
)

assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes dans BASE_GROUPES : "
    f"{colonnes_manquantes}. "
    "Exécuter d'abord les cellules 42 à 47 corrigées."
)


# ------------------------------------------------------------
# 2. Fonction de remédiation par groupe
# ------------------------------------------------------------

def construire_remediation(row):

    lacunes = row[
        "LACUNES_PRIORITAIRES"
    ]

    # --------------------------------------------------------
    # Aucun besoin ciblé détecté
    # --------------------------------------------------------

    if pd.isna(lacunes):

        return (
            "Aucune lacune prioritaire identifiée au niveau du groupe ; "
            "poursuivre le programme commun du parcours."
        )


    lacunes = str(
        lacunes
    ).strip()


    if (
        lacunes == ""
        or lacunes.lower() == "nan"
        or lacunes in {
            "Aucune",
            "Aucune_lacune_T1",
            "Aucune lacune T1 détectée"
        }
    ):

        return (
            "Aucune lacune prioritaire identifiée au niveau du groupe ; "
            "poursuivre le programme commun du parcours."
        )


    # --------------------------------------------------------
    # Lacunes communes / prioritaires identifiées
    # --------------------------------------------------------

    return (
        "Renforcer prioritairement les compétences suivantes : "
        f"{lacunes}. "
        "Sélectionner dans le programme commun des ressources "
        "de renforcement correspondantes."
    )


# ------------------------------------------------------------
# 3. Application
# ------------------------------------------------------------

BASE_GROUPES[
    "REMEDIATION_LACUNES"
] = (
    BASE_GROUPES
    .apply(
        construire_remediation,
        axis=1
    )
)


# ------------------------------------------------------------
# 4. Vérification :
#    une même remédiation doit être associée à un même groupe
# ------------------------------------------------------------

controle_remediation = (
    BASE_GROUPES
    .groupby(
        "ID_GROUPE"
    )
    ["REMEDIATION_LACUNES"]
    .nunique()
)


assert (
    controle_remediation <= 1
).all(), (
    "Un même groupe possède plusieurs recommandations "
    "de remédiation différentes."
)


# ------------------------------------------------------------
# 5. Affichage synthétique
# ------------------------------------------------------------

resume_remediation = (
    BASE_GROUPES[
        [
            "ID_GROUPE",
            "T1__PARCOURS",
            "NIVEAU_PARCOURS",
            "LACUNES_PRIORITAIRES",
            "REMEDIATION_LACUNES"
        ]
    ]
    .drop_duplicates(
        subset=[
            "ID_GROUPE"
        ]
    )
    .sort_values(
        "ID_GROUPE"
    )
    .reset_index(
        drop=True
    )
)


display(
    resume_remediation
)


print(
    "✅ Remédiation construite pour",
    resume_remediation[
        "ID_GROUPE"
    ].nunique(),
    "groupes."
)

,ID_GROUPE,T1__PARCOURS,NIVEAU_PARCOURS,LACUNES_PRIORITAIRES,REMEDIATION_LACUNES
0,G001,BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,Renforcer prioritairement les compétences suiv...
1,G002,BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,Renforcer prioritairement les compétences suiv...
2,G003,BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,Renforcer prioritairement les compétences suiv...
3,G004,DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Renforcer prioritairement les compétences suiv...
4,G005,DA,Debutant,Power Query (fusion),Renforcer prioritairement les compétences suiv...
5,G006,DA,Intermediaire,Power Pivot (schéma en étoile) | Power Query (...,Renforcer prioritairement les compétences suiv...
6,G007,DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Renforcer prioritairement les compétences suiv...
7,G008,DS,Debutant,Machine Learning (généralisation) | Pandas (jo...,Renforcer prioritairement les compétences suiv...
8,G009,DS,Intermediaire,Machine Learning (déséquilibre) | Statistiques...,Renforcer prioritairement les compétences suiv...
9,G010,IA,Avance,Open WebUI (Skills) | LLM (paramètres) | Open ...,Renforcer prioritairement les compétences suiv...


✅ Remédiation construite pour 12 groupes.


# Cellule 51 — Renforcement des prérequis

In [52]:
# ============================================================
# CELLULE 51 — Prérequis
# ============================================================

def adaptation_prerequis(prerequis):

    if pd.isna(prerequis):
        return "Prerequis a verifier"

    if prerequis == "Aucun":
        return "Remediation prerequis prioritaire"

    if prerequis == "Debutant":
        return "Renforcement des prerequis"

    if prerequis == "Intermediaire":
        return "Rappel cible si necessaire"

    if prerequis == "Avance":
        return "Pas de renforcement prioritaire"

    return "Prerequis a verifier"


BASE_GROUPES[
    "ADAPTATION_PREREQUIS"
] = (
    BASE_GROUPES[
        "PREREQUIS_DOMINANT"
    ]
    .apply(
        adaptation_prerequis
    )
)

# Cellule 52 — Fiche pédagogique finale par groupe

In [53]:
# ============================================================
# CELLULE 52 — Recommandation par groupe
# ============================================================

RECOMMANDATIONS_CONTENU = (

    BASE_GROUPES

    .groupby(
        "ID_GROUPE",
        as_index=False
    )

    .agg(
        PARCOURS=(
            "T1__PARCOURS",
            "first"
        ),

        PROGRAMME_COMMUN=(
            "PROGRAMME_COMMUN",
            "first"
        ),

        NIVEAU=(
            "NIVEAU_PARCOURS",
            "first"
        ),

        STATUT_NIVEAU=(
            "STATUT_NIVEAU",
            "first"
        ),

        N_APPRENANTS=(
            ID,
            "nunique"
        ),

        LACUNE_PRINCIPALE=(
            "LACUNES_PRIORITAIRES",
            "first"
        ),

        PREREQUIS_LIMITANT=(
            "PREREQUIS_DOMINANT",
            "first"
        ),

        OBJECTIFS_DOMINANTS=(
            "OBJECTIFS_DOMINANTS",
            "first"
        ),

        TYPE_ACCOMPAGNEMENT=(
            "TYPE_ACCOMPAGNEMENT",
            "first"
        ),

        EXPLICATION=(
            "EXPLICATION",
            "first"
        ),

        EXERCICES=(
            "EXERCICES",
            "first"
        ),

        REMEDIATION_LACUNES=(
            "REMEDIATION_LACUNES",
            "first"
        ),

        ADAPTATION_PREREQUIS=(
            "ADAPTATION_PREREQUIS",
            "first"
        )
    )
)


RECOMMANDATIONS_CONTENU[
    "SOURCE_LACUNES"
] = "T1"


display(
    RECOMMANDATIONS_CONTENU
)

,ID_GROUPE,PARCOURS,PROGRAMME_COMMUN,NIVEAU,STATUT_NIVEAU,N_APPRENANTS,LACUNE_PRINCIPALE,PREREQUIS_LIMITANT,OBJECTIFS_DOMINANTS,TYPE_ACCOMPAGNEMENT,EXPLICATION,EXERCICES,REMEDIATION_LACUNES,ADAPTATION_PREREQUIS,SOURCE_LACUNES
0,G001,BI,PROGRAMME_BI,Avance,EXPLORATOIRE_KMEANS_OOF,11,Modèle en étoile | Relations Power BI | Cardin...,Aucun,Data Analyst | Business Analyst,Approfondissement,Synthetique avec approfondissements,"Complexes, challenges et mini-projets",Renforcer prioritairement les compétences suiv...,Remediation prerequis prioritaire,T1
1,G002,BI,PROGRAMME_BI,Debutant,EXPLORATOIRE_KMEANS_OOF,20,Typage des clés | Nettoyage de cellules fusion...,Aucun,Data Analyst | Entrepreneur,Renforce,Detaillee et progressive,Guides puis progressifs,Renforcer prioritairement les compétences suiv...,Remediation prerequis prioritaire,T1
2,G003,BI,PROGRAMME_BI,Intermediaire,EXPLORATOIRE_KMEANS_OOF,11,Typage des clés | Relations plusieurs-à-plusie...,Aucun,Entrepreneur | Data Analyst,Standard cible,Rappels et consolidation,Application et cas pratiques,Renforcer prioritairement les compétences suiv...,Remediation prerequis prioritaire,T1
3,G004,DA,PROGRAMME_DA,Avance,EXPLORATOIRE_KMEANS_OOF,6,Power Query (évaluation paresseuse) | Power Pi...,Debutant,Entrepreneur | Developpeur Python / IA,Approfondissement,Synthetique avec approfondissements,"Complexes, challenges et mini-projets",Renforcer prioritairement les compétences suiv...,Renforcement des prerequis,T1
4,G005,DA,PROGRAMME_DA,Debutant,EXPLORATOIRE_KMEANS_OOF,5,Power Query (fusion),Debutant,Data Analyst | Non defini,Renforce,Detaillee et progressive,Guides puis progressifs,Renforcer prioritairement les compétences suiv...,Renforcement des prerequis,T1
5,G006,DA,PROGRAMME_DA,Intermediaire,EXPLORATOIRE_KMEANS_OOF,6,Power Pivot (schéma en étoile) | Power Query (...,Debutant,Entrepreneur | Business Analyst,Standard cible,Rappels et consolidation,Application et cas pratiques,Renforcer prioritairement les compétences suiv...,Renforcement des prerequis,T1
6,G007,DS,PROGRAMME_DS,Avance,EXPLORATOIRE_KMEANS_OOF,8,Machine Learning (déséquilibre) | Pandas (join...,Debutant,Developpeur Python / IA | Data Analyst,Approfondissement,Synthetique avec approfondissements,"Complexes, challenges et mini-projets",Renforcer prioritairement les compétences suiv...,Renforcement des prerequis,T1
7,G008,DS,PROGRAMME_DS,Debutant,EXPLORATOIRE_KMEANS_OOF,13,Machine Learning (généralisation) | Pandas (jo...,Debutant,Data Analyst | Entrepreneur,Renforce,Detaillee et progressive,Guides puis progressifs,Renforcer prioritairement les compétences suiv...,Renforcement des prerequis,T1
8,G009,DS,PROGRAMME_DS,Intermediaire,EXPLORATOIRE_KMEANS_OOF,5,Machine Learning (déséquilibre) | Statistiques...,Intermediaire,Data Analyst | Entrepreneur,Standard cible,Rappels et consolidation,Application et cas pratiques,Renforcer prioritairement les compétences suiv...,Rappel cible si necessaire,T1
9,G010,IA,PROGRAMME_IA,Avance,EXPLORATOIRE_KMEANS_OOF,8,Open WebUI (Skills) | LLM (paramètres) | Open ...,Debutant,Developpeur Python / IA | Entrepreneur,Approfondissement,Synthetique avec approfondissements,"Complexes, challenges et mini-projets",Renforcer prioritairement les compétences suiv...,Renforcement des prerequis,T1


# Cellule 53 — Requête RAG corrigée
Le RAG doit chercher dans le programme du parcours, pas inventer un autre programme.

In [54]:
# ============================================================
# CELLULE 53 — Requête RAG par groupe
# ============================================================

# ------------------------------------------------------------
# 1. Vérification des colonnes nécessaires
# ------------------------------------------------------------

colonnes_requises = {
    "ID_GROUPE",
    "PARCOURS",
    "PROGRAMME_COMMUN",
    "NIVEAU",
    "STATUT_NIVEAU",
    "N_APPRENANTS",
    "LACUNE_PRINCIPALE",
    "PREREQUIS_LIMITANT",
    "OBJECTIFS_DOMINANTS",
    "TYPE_ACCOMPAGNEMENT",
    "EXPLICATION",
    "EXERCICES",
    "REMEDIATION_LACUNES",
    "ADAPTATION_PREREQUIS",
    "SOURCE_LACUNES"
}


colonnes_manquantes = (
    colonnes_requises
    -
    set(RECOMMANDATIONS_CONTENU.columns)
)


assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes dans RECOMMANDATIONS_CONTENU : "
    f"{colonnes_manquantes}. "
    "Exécuter d'abord la Cellule 52 corrigée."
)


# ------------------------------------------------------------
# 2. Fonction de nettoyage pour éviter 'nan' dans le texte
# ------------------------------------------------------------

def texte_valide(valeur, defaut="Non renseigné"):

    if pd.isna(valeur):
        return defaut

    texte = str(valeur).strip()

    if (
        texte == ""
        or texte.lower() == "nan"
    ):
        return defaut

    return texte


# ------------------------------------------------------------
# 3. Construction de la requête RAG
# ------------------------------------------------------------

def construire_requete_rag(row):

    programme = texte_valide(
        row["PROGRAMME_COMMUN"]
    )

    parcours = texte_valide(
        row["PARCOURS"]
    )

    niveau = texte_valide(
        row["NIVEAU"]
    )

    statut_niveau = texte_valide(
        row["STATUT_NIVEAU"]
    )

    lacunes = texte_valide(
        row["LACUNE_PRINCIPALE"],
        defaut="Aucune lacune prioritaire identifiée"
    )

    prerequis = texte_valide(
        row["PREREQUIS_LIMITANT"],
        defaut="Aucun prérequis limitant identifié"
    )

    objectifs = texte_valide(
        row["OBJECTIFS_DOMINANTS"],
        defaut="Objectifs non renseignés"
    )

    accompagnement = texte_valide(
        row["TYPE_ACCOMPAGNEMENT"]
    )

    explication = texte_valide(
        row["EXPLICATION"]
    )

    exercices = texte_valide(
        row["EXERCICES"]
    )

    remediation = texte_valide(
        row["REMEDIATION_LACUNES"]
    )

    adaptation_prerequis = texte_valide(
        row["ADAPTATION_PREREQUIS"]
    )

    source_lacunes = texte_valide(
        row["SOURCE_LACUNES"],
        defaut="T1"
    )


    return (
        f"Programme commun : {programme}. "
        f"Parcours : {parcours}. "
        f"Niveau exploratoire du groupe : {niveau}. "
        f"Statut du niveau : {statut_niveau}. "
        f"Conserver strictement le même programme, "
        f"les mêmes chapitres et les mêmes objectifs pédagogiques "
        f"du parcours. "
        f"Ne pas créer un nouveau programme. "
        f"Adapter uniquement l'accompagnement pédagogique. "
        f"Lacunes prioritaires du groupe : {lacunes}. "
        f"Source des lacunes : {source_lacunes}. "
        f"Prérequis limitant dominant : {prerequis}. "
        f"Objectifs dominants du groupe : {objectifs}. "
        f"Type d'accompagnement : {accompagnement}. "
        f"Explications attendues : {explication}. "
        f"Exercices attendus : {exercices}. "
        f"Remédiation : {remediation}. "
        f"Adaptation des prérequis : {adaptation_prerequis}."
    )


# ------------------------------------------------------------
# 4. Construire la requête pour chaque groupe
# ------------------------------------------------------------

RECOMMANDATIONS_CONTENU[
    "REQUETE_RAG"
] = (
    RECOMMANDATIONS_CONTENU
    .apply(
        construire_requete_rag,
        axis=1
    )
)


# ------------------------------------------------------------
# 5. Contrôles
# ------------------------------------------------------------

assert (
    RECOMMANDATIONS_CONTENU[
        "REQUETE_RAG"
    ]
    .notna()
    .all()
)


assert (
    RECOMMANDATIONS_CONTENU[
        "REQUETE_RAG"
    ]
    .str.len()
    > 0
).all()


# ------------------------------------------------------------
# 6. Affichage
# ------------------------------------------------------------

display(
    RECOMMANDATIONS_CONTENU[
        [
            "ID_GROUPE",
            "PARCOURS",
            "PROGRAMME_COMMUN",
            "NIVEAU",
            "LACUNE_PRINCIPALE",
            "PREREQUIS_LIMITANT",
            "OBJECTIFS_DOMINANTS",
            "REQUETE_RAG"
        ]
    ]
    .head(10)
)


print(
    "✅ Requêtes RAG construites pour",
    len(RECOMMANDATIONS_CONTENU),
    "groupes."
)

,ID_GROUPE,PARCOURS,PROGRAMME_COMMUN,NIVEAU,LACUNE_PRINCIPALE,PREREQUIS_LIMITANT,OBJECTIFS_DOMINANTS,REQUETE_RAG
0,G001,BI,PROGRAMME_BI,Avance,Modèle en étoile | Relations Power BI | Cardin...,Aucun,Data Analyst | Business Analyst,Programme commun : PROGRAMME_BI. Parcours : BI...
1,G002,BI,PROGRAMME_BI,Debutant,Typage des clés | Nettoyage de cellules fusion...,Aucun,Data Analyst | Entrepreneur,Programme commun : PROGRAMME_BI. Parcours : BI...
2,G003,BI,PROGRAMME_BI,Intermediaire,Typage des clés | Relations plusieurs-à-plusie...,Aucun,Entrepreneur | Data Analyst,Programme commun : PROGRAMME_BI. Parcours : BI...
3,G004,DA,PROGRAMME_DA,Avance,Power Query (évaluation paresseuse) | Power Pi...,Debutant,Entrepreneur | Developpeur Python / IA,Programme commun : PROGRAMME_DA. Parcours : DA...
4,G005,DA,PROGRAMME_DA,Debutant,Power Query (fusion),Debutant,Data Analyst | Non defini,Programme commun : PROGRAMME_DA. Parcours : DA...
5,G006,DA,PROGRAMME_DA,Intermediaire,Power Pivot (schéma en étoile) | Power Query (...,Debutant,Entrepreneur | Business Analyst,Programme commun : PROGRAMME_DA. Parcours : DA...
6,G007,DS,PROGRAMME_DS,Avance,Machine Learning (déséquilibre) | Pandas (join...,Debutant,Developpeur Python / IA | Data Analyst,Programme commun : PROGRAMME_DS. Parcours : DS...
7,G008,DS,PROGRAMME_DS,Debutant,Machine Learning (généralisation) | Pandas (jo...,Debutant,Data Analyst | Entrepreneur,Programme commun : PROGRAMME_DS. Parcours : DS...
8,G009,DS,PROGRAMME_DS,Intermediaire,Machine Learning (déséquilibre) | Statistiques...,Intermediaire,Data Analyst | Entrepreneur,Programme commun : PROGRAMME_DS. Parcours : DS...
9,G010,IA,PROGRAMME_IA,Avance,Open WebUI (Skills) | LLM (paramètres) | Open ...,Debutant,Developpeur Python / IA | Entrepreneur,Programme commun : PROGRAMME_IA. Parcours : IA...


✅ Requêtes RAG construites pour 12 groupes.


La différence porte uniquement sur l’accompagnement, les explications, les exercices, les remédiations et les prérequis, pas sur le programme lui-même.

In [55]:
# ============================================================
# EXPORT PASSERELLE 05 → 06
# ============================================================

# ------------------------------------------------------------
# 1. Vérifications des objets
# ------------------------------------------------------------

assert "RECOMMANDATIONS_CONTENU" in globals(), (
    "RECOMMANDATIONS_CONTENU n'existe pas. "
    "Exécuter d'abord la Cellule 52 puis la Cellule 53."
)

assert "BASE_GROUPES" in globals(), (
    "BASE_GROUPES n'existe pas."
)

assert "groupes_pedagogiques" in globals(), (
    "groupes_pedagogiques n'existe pas."
)


# ------------------------------------------------------------
# 2. Vérification du schéma de la passerelle principale
# ------------------------------------------------------------

colonnes_recommandations_requises = {
    "ID_GROUPE",
    "PARCOURS",
    "PROGRAMME_COMMUN",
    "NIVEAU",
    "STATUT_NIVEAU",
    "N_APPRENANTS",
    "LACUNE_PRINCIPALE",
    "PREREQUIS_LIMITANT",
    "OBJECTIFS_DOMINANTS",
    "TYPE_ACCOMPAGNEMENT",
    "EXPLICATION",
    "EXERCICES",
    "REMEDIATION_LACUNES",
    "ADAPTATION_PREREQUIS",
    "SOURCE_LACUNES",
    "REQUETE_RAG"
}


colonnes_manquantes = (
    colonnes_recommandations_requises
    -
    set(RECOMMANDATIONS_CONTENU.columns)
)


assert len(colonnes_manquantes) == 0, (
    "Colonnes manquantes dans RECOMMANDATIONS_CONTENU : "
    f"{colonnes_manquantes}"
)


# ------------------------------------------------------------
# 3. Contrôles de cohérence
# ------------------------------------------------------------

assert (
    RECOMMANDATIONS_CONTENU[
        "ID_GROUPE"
    ]
    .notna()
    .all()
)


assert (
    RECOMMANDATIONS_CONTENU[
        "ID_GROUPE"
    ]
    .is_unique
), (
    "ID_GROUPE doit être unique dans RECOMMANDATIONS_CONTENU."
)


assert (
    BASE_GROUPES[
        "ID_GROUPE"
    ]
    .notna()
    .all()
)


assert (
    BASE_GROUPES[
        ID
    ]
    .notna()
    .all()
)


assert (
    RECOMMANDATIONS_CONTENU[
        "REQUETE_RAG"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 4. Nettoyage de quelques anciennes colonnes éventuelles
# ------------------------------------------------------------

colonnes_obsoletes = [
    "T2_PREDIT",
    "NIVEAU_PREDIT",
    "CLUSTER_NIVEAU"
]


BASE_GROUPES_EXPORT = (
    BASE_GROUPES
    .drop(
        columns=[
            col
            for col in colonnes_obsoletes
            if col in BASE_GROUPES.columns
        ],
        errors="ignore"
    )
    .copy()
)


# ------------------------------------------------------------
# 5. Fichiers de sortie
# ------------------------------------------------------------

fichier_recommandations = (
    TABLE_DIR
    / "recommandations_contenu_groupes.csv"
)


fichier_base_groupes = (
    TABLE_DIR
    / "base_groupes_pedagogiques.csv"
)


fichier_groupes = (
    TABLE_DIR
    / "groupes_pedagogiques.csv"
)


# ------------------------------------------------------------
# 6. Export CSV
# ------------------------------------------------------------

RECOMMANDATIONS_CONTENU.to_csv(
    fichier_recommandations,
    index=False,
    encoding="utf-8-sig"
)


BASE_GROUPES_EXPORT.to_csv(
    fichier_base_groupes,
    index=False,
    encoding="utf-8-sig"
)


groupes_pedagogiques.to_csv(
    fichier_groupes,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 7. Vérification après écriture
# ------------------------------------------------------------

assert fichier_recommandations.exists()
assert fichier_base_groupes.exists()
assert fichier_groupes.exists()


# ------------------------------------------------------------
# 8. Résumé
# ------------------------------------------------------------

print("=" * 70)
print("PASSERELLE 05 → 06")
print("=" * 70)

print(
    "Groupes recommandés :",
    len(RECOMMANDATIONS_CONTENU)
)

print(
    "Apprenants dans BASE_GROUPES :",
    BASE_GROUPES_EXPORT[
        ID
    ].nunique()
)

print(
    "Groupes pédagogiques :",
    groupes_pedagogiques[
        "ID_GROUPE"
    ].nunique()
)


print(
    "\nFichiers exportés :"
)

print(
    "-",
    fichier_recommandations
)

print(
    "-",
    fichier_base_groupes
)

print(
    "-",
    fichier_groupes
)


print(
    "\n✅ Passerelle 05 → 06 exportée avec succès."
)

PASSERELLE 05 → 06
Groupes recommandés : 12
Apprenants dans BASE_GROUPES : 125
Groupes pédagogiques : 12

Fichiers exportés :
- /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/05_groupes_recommandation/tables/recommandations_contenu_groupes.csv
- /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/05_groupes_recommandation/tables/base_groupes_pedagogiques.csv
- /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/05_groupes_recommandation/tables/groupes_pedagogiques.csv

✅ Passerelle 05 → 06 exportée avec succès.
